# API reference

Every public name in the `mboostrs` Python package, grouped by task, with its signature and defaults, a one-line description, and for most entries a short copyable example.

For argument descriptions, return values and raised errors, open the symbol's page in the [Python API manual](../api/index.md), for example [`glmboost`](../api/functions/glmboost.md). In the snippets below, `mb` refers to `mboostrs` and `fit` refers to a fitted model.

## Fitting

`fit(data, response, ...)` selects and fits a specification and returns a `Fit`: the fitted model plus its decision trail, a JSON-serialisable record of the data, the folds, each candidate change with its out-of-fold score, and the final specification. `glmboost` and `gamboost` fit a supplied formula; `glmboostLSS` and `gamboostLSS` fit distributional models (LSS: location, scale and shape), in which each distribution parameter has its own additive predictor. Arguments after `*` in a signature are keyword-only.

| Name | What it is |
|---|---|
| `fit(data: Any, response: str, *, family: Any = None, folds: Any = None, groups: Any = None, time: Any = None, seed: int = 0, budget: str = 'standard', uncertainty: str = 'none', na_action: str \| None = None, **kwargs: Any) -> mboostrs._procedure._fit.Fit` | Fit one model and return it with the trail of how it was chosen. |
| `gam(formula, data, *, family='gaussian', weights=None, offset=None, na_action='error', method='REML')` | Fit a penalized GAM by REML from a formula, with no boosting. |
| `gamboost(formula: str, data, *, family=None, control=None, weights=None, offset=None, learners: list \| None = None, env: dict \| None = None, subsample: float \| None = None, subsample_seed: int = 0, na_action: str = 'error', bins: int \| None = None, weight_policy: str = 'upstream') -> mboostrs.gamboost.GAMBoostModel` | Fit a component-wise boosted additive model. |
| `gamboostLSS(formula, data, *, families=None, family=None, control=None, offset=None, mstop=None, weights=None, method='cyclic', bins=None, env=None, weight_policy='upstream', stabilization=None, nu=None, na_action='error', response_scale='auto')` | Fit a distributional additive model with linear, spline, and structured learners. |
| `glmboost(formula: str, data, *, family=None, control=None, weights=None, offset=None, center: bool = True, subsample: float \| None = None, subsample_seed: int = 0, na_action: str = 'error', weight_policy: str = 'upstream') -> mboostrs.glmboost.GLMBoostModel` | Fit a component-wise boosted linear model. |
| `glmboostLSS(formula, data, *, families=None, family=None, control=None, offset=None, mstop=None, method='cyclic', weights=None, env=None, weight_policy='upstream', stabilization=None, nu=None, na_action='error', response_scale='auto')` | Fit a linear GAMLSS by component-wise boosting. |

**`fit`**

```python
df = mb.datasets.bodyfat()
m = mb.fit(df, "DEXfat", seed=0)
m.trail.validate()  # schema-backed evidence of how m.model was chosen
```

**`gam`**

```python
body = mb.datasets.bodyfat()
fit = mb.gam("DEXfat ~ bbs(hipcirc, knots=6) + bbs(waistcirc, knots=6)", data=body)
fit.contrast("hipcirc", 90.0, 110.0, data=body).estimate
```

**`gamboost`**

```python
df = mb.datasets.bodyfat()
fit = mb.gamboost("DEXfat ~ bbs(hipcirc, df=4) + bols(waistcirc)",
                   data=df)
```

**`gamboostLSS`**

```python
df = mb.datasets.heteroscedastic_lss()
fit = mb.gamboostLSS("y ~ bbs(x1, df=3) + bols(x2)", data=df,
                     families=mb.GaussianLSS())
```

**`glmboost`**

```python
df = mb.datasets.bodyfat()
fit = mb.glmboost("DEXfat ~ waistcirc + hipcirc", data=df)
```

**`glmboostLSS`**

```python
df = mb.datasets.heteroscedastic_lss()
fit = mb.glmboostLSS("y ~ bols(x1) + bols(x2)", data=df,
                     families=mb.GaussianLSS())
```


## Control

Set the maximum iterations and update step length.

| Name | What it is |
|---|---|
| `BoostControl(mstop: int, nu: float, risk: str, trace: bool, step: str = 'fixed', step_iterations: int = 4)` | How long to boost, and how far each step goes. |
| `boost_control(mstop: int = 100, nu: float = 0.1, risk: str = 'inbag', trace: bool = False, step: str = 'fixed', step_iterations: int = 4) -> mboostrs._control.BoostControl` | The control record a fitting function takes. |

**`boost_control`**

```python
ctrl = mb.boost_control(mstop=200, nu=0.1)
fit = mb.glmboost("DEXfat ~ waistcirc",
                   data=mb.datasets.bodyfat(), control=ctrl)
```


## Fitted models

Fitted-model classes and stopping budgets.

| Name | What it is |
|---|---|
| `Fit` | A fitted model wrapped with the trail that produced it. |
| `GAMBoostModel` | Fitted generalized additive boosting model. |
| `GLMBoostModel` | Fitted gradient boosting model. |
| `LSSModel` | Fitted gamboostLSS/glmboostLSS model (one predictor per parameter). |
| `StoppingBudget` | How far a fitted model was boosted, in the same shape for every model. |

**`Fit`**

```python
df = mb.datasets.bodyfat()
m = mb.fit(df, "DEXfat", seed=0)
m.model, m.trail, m.stability  # the fit, its trail, and the uncertainty step
```

**`StoppingBudget`**

```python
budget = fit.stopping_budget()
budget.scheme, budget.total, budget.per_parameter
```


## Families

Each family defines the loss and response encoding used by the fit. The *Python fit* and *R fit* columns give the evidence the capability manifest (`mb.family_capabilities()`) records for fitting in each frontend: `verified` means the fit is compared with a reference implementation or reference value within a stated tolerance; `smoke` means tests check that the fit runs and satisfies stated invariants, with no reference to compare against; `intentional-deviation` means the tested behaviour differs deliberately from the R `mboost` or `gamboostLSS` counterpart; `unverified` means the manifest makes no correctness claim. The same columns appear in the distributional-family and base-learner tables. The [capabilities](capabilities.ipynb) page lists every status value, and the [R parity matrix](r_parity_matrix.ipynb) lists the comparisons with R.

| Name | What it is | Python fit | R fit |
|---|---|---|---|
| `AUC()` | Smoothed 1 - AUC ranking loss (mboost::AUC()). | verified | unverified |
| `AdaExp()` | Exponential loss, `exp(-(2y - 1) f)` — the AdaBoost loss. | verified | unverified |
| `Binomial()` | Binomial log-likelihood with a logit link, for a two-class response. | verified | unverified |
| `Cindex(sigma=0.1)` | Uno's IPCW-weighted concordance ("C-index") loss (mboost::Cindex()). | verified | verified |
| `CoxPH()` | Cox partial likelihood for right-censored survival data. | verified | unverified |
| `Expectile(tau=0.5)` | Asymmetric squared error loss for expectile regression. | verified | unverified |
| `GammaReg()` | Gamma log-likelihood with a log link, for a positive continuous response. | verified | unverified |
| `GaussReg()` | Squared error loss with an identity link, without R's `mboost` equivalent. | verified | unverified |
| `Gaussian()` | Squared error loss, `0.5 * (y - f)**2`, with an identity link. | verified | verified |
| `Gehan()` | Gehan rank loss for AFT survival models (mboost::Gehan()). | intentional-deviation | intentional-deviation |
| `HingeLoss(alpha_cost=0.5)` | Hinge loss, `max(0, 1 - y f)` — the support vector loss. | unverified | unverified |
| `Huber(delta=None)` | Huber's robust loss: quadratic near zero, linear in the tails. | verified | unverified |
| `Hurdle()` | The count part of a hurdle model: a zero-truncated negative binomial. | verified | verified |
| `Laplace(response_scale='auto')` | Absolute error loss, `\|y - f\|`, with an identity link. | verified | unverified |
| `Loglog()` | Log-logistic accelerated-failure-time model for right-censored data. | verified | unverified |
| `Lognormal()` | Log-normal accelerated-failure-time model for right-censored data. | unverified | unverified |
| `Multinomial(n_classes=None)` | Multinomial logistic likelihood for an unordered K-class response. | verified | unverified |
| `NBinomial(**kwargs)` | Negative binomial log-likelihood with a log link, for overdispersed counts. | verified | unverified |
| `Poisson()` | Poisson log-likelihood with a log link, for counts. | verified | unverified |
| `PropOdds(n_levels=None, levels=None)` | Proportional-odds cumulative model for an ordered categorical response. | verified | unverified |
| `Quantile(tau=0.5, qoffset=None, response_scale='auto')` | Check (pinball) loss for quantile regression. | verified | unverified |
| `Weibull()` | Weibull accelerated-failure-time model for right-censored data. | verified | unverified |

**`AUC`**

```python
family = mb.AUC()  # y: two-level factor or {-1, +1}; ranking objective
```

**`AdaExp`**

```python
family = mb.AdaExp()  # y in {0, 1} or {-1, +1}; exponential-loss classification
```

**`Binomial`**

```python
family = mb.Binomial()  # y in {0, 1} (or {-1, +1}, or [0, 1] proportions)
```

**`Cindex`**

```python
family = mb.Cindex()  # Surv(time, status); concordance
```

**`CoxPH`**

```python
family = mb.CoxPH()  # Surv(time, status); Cox partial likelihood
```

**`Expectile`**

```python
family = mb.Expectile()  # y: any real value; expectile set at construction
```

**`GammaReg`**

```python
family = mb.GammaReg()  # y: strictly positive
```

**`GaussReg`**

```python
family = mb.GaussReg()  # y: any real value; the same model as Gaussian
```

**`Gaussian`**

```python
family = mb.Gaussian()  # y: any real value
```

**`Gehan`**

```python
family = mb.Gehan()  # Surv(time, status); AFT rank loss
```

**`HingeLoss`**

```python
family = mb.HingeLoss()  # y in {-1, +1}; margin classification
```

**`Huber`**

```python
family = mb.Huber()  # y: any real value; robust to outliers
```

**`Hurdle`**

```python
family = mb.Hurdle()  # y: positive counts only; the zero part is a separate binary model
```

**`Laplace`**

```python
family = mb.Laplace()  # y: any real value; median regression
```

**`Loglog`**

```python
family = mb.Loglog()  # Surv(time, status); parametric survival
```

**`Lognormal`**

```python
family = mb.Lognormal()  # Surv(time, status); parametric survival
```

**`Multinomial`**

```python
family = mb.Multinomial()  # y: integer class codes or factor; softmax over classes
```

**`NBinomial`**

```python
family = mb.NBinomial()  # y: counts; overdispersed relative to Poisson
```

**`Poisson`**

```python
family = mb.Poisson()  # y: counts, non-negative integers
```

**`PropOdds`**

```python
family = mb.PropOdds()  # y: ordered factor; proportional-odds
```

**`Quantile`**

```python
family = mb.Quantile()  # y: any real value; quantile set at construction
```

**`Weibull`**

```python
family = mb.Weibull()  # Surv(time, status); parametric survival
```


## Distributional families

Each distribution parameter has its own additive predictor.

| Name | What it is | Python fit | R fit |
|---|---|---|---|
| `BCCGLSS(stabilization: str = 'none')` | Box-Cox-Cole-Green family: `mu` (log link, median), `sigma` (log link, scale), `nu` (identity link, Box-Cox power). Matches R `gamboostLSS::BCCGLSS()` (wrapping gamlss.dist `BCCG`). Requires `y > 0`. | unverified | unverified |
| `BCPELSS(stabilization: str = 'none')` | Box-Cox power-exponential family with four fitted parameters. | verified | verified |
| `BCTLSS(stabilization: str = 'none')` | Box-Cox t family with `mu`, `sigma`, `nu`, and `tau`. | verified | verified |
| `BetaLSS(stabilization: str = 'none')` | Beta location-precision family: `mu` (logit link, mean in (0,1)), `phi` (log link, precision). Matches R `gamboostLSS::BetaLSS()`. Requires `0 < y < 1`. | verified | verified |
| `BinomialLSS(stabilization: str = 'none')` | Binomial location family: `mu` (logit link, probability). Matches R `gamboostLSS::BinomialLSS()` (wrapping gamlss.dist `BI`). Requires `y` in `[0, 1]` (proportions). | unverified | unverified |
| `DirichletLSS(K: int, stabilization: str = 'none')` | Dirichlet family for compositional responses: `K` concentration parameters `alpha_k = exp(eta_k)` (log links). The response is a `K`-column composition specified as `cbind(y1, ..., yK) ~ ...` (each row sums to 1). Matches R `gamboostLSS::DirichletLSS(K)`. Requires `0 < y < 1`. | verified | verified |
| `GammaLSS(stabilization: str = 'none')` | Gamma location-shape family: `mu` (log link, mean), `sigma` (log link, shape). Matches R `gamboostLSS::GammaLSS()`. Requires `y > 0`. | verified | verified |
| `GaussianKnownVarLSS(stabilization: str = 'none')` | Normal location-scale family for outcomes that carry a known sampling variance: `v_i = s2_i + exp(eta_sigma_i)`, the known sampling variance plus a boosted *excess* variance component. | smoke | smoke |
| `GaussianLSS(stabilization: str = 'none')` | Normal location-scale family: `mu` (identity link), `sigma` (log link). | verified | verified |
| `GeometricLSS(stabilization: str = 'none')` | `\text{Geom}(\pi)` geometric location family: `mu` (log link, mean). Matches R `gamboostLSS::GeometricLSS()` (wrapping gamlss.dist `GEOM`). Requires non-negative integer counts. | unverified | unverified |
| `GumbelLSS(stabilization: str = 'none')` | Gumbel location-scale family for maxima — the standard extreme-value type I: `mu` (identity link, location), `sigma` (log link, scale). | unverified | unverified |
| `InverseGaussianLSS(stabilization: str = 'none')` | Inverse Gaussian family: `mu` (log link, mean), `sigma` (log link, dispersion). Requires `y > 0`. Matches `gamlss.dist::IG()`. | unverified | unverified |
| `JSULSS(stabilization: str = 'none')` | Johnson's SU family: `mu` (identity link, location), `sigma` (log link, scale), `nu` (identity link, skewness), `tau` (log link, kurtosis). Matches R `gamboostLSS::JSULSS()` (wrapping gamlss.dist `JSU`). | unverified | unverified |
| `LogLogLSS(stabilization: str = 'none')` | Log-logistic AFT family. Takes a right-censored `Surv(time, status)` response, or a plain positive response treated as uncensored. | intentional-deviation | intentional-deviation |
| `LogNormalLSS(stabilization: str = 'none')` | Log-normal AFT family. Matches R `gamboostLSS::LogNormalLSS()` for a right-censored `Surv(time, status)` response. | verified | verified |
| `LogisticLSS(stabilization: str = 'none')` | Logistic family: `mu` (identity link, location), `sigma` (log link, scale). `y` unrestricted. Matches `gamlss.dist::LO()`. | unverified | unverified |
| `NBinomialLSS(stabilization: str = 'none')` | Negative-binomial family: `mu` (log link, mean), `sigma` (log link, size/dispersion). Matches R `gamboostLSS::NBinomialLSS()`. Requires non-negative integer counts. | verified | verified |
| `NegativeBinomialLSS(stabilization: str = 'none')` | Negative-binomial location-scale family: `mu` (log link, mean), `sigma` (log link, dispersion), variance `mu + sigma * mu**2`. This is gamlss.dist `NBI`: `pdf` equals `dNBI(y, mu, sigma)`. gamboostLSS has no family of this name; its `NBinomialLSS` (`NBinomialLSS` here) is the same distribution with `sigma` naming the size `1/sigma`. Requires non-negative integer counts. | unverified | unverified |
| `PIGLSS(stabilization: str = 'none')` | Poisson-inverse-Gaussian family: `mu` (log link, mean), `sigma` (log link, dispersion). Matches R `gamboostLSS::PIGLSS()` (wrapping gamlss.dist `PIG`). Requires non-negative integer counts. | smoke | unverified |
| `Pareto2LSS(stabilization: str = 'none')` | Pareto type-2 (Lomax) family: `mu` (log link, scale), `sigma` (log link, tail shape). The density is `sigma * mu**sigma / (y + mu)**(sigma + 1)` for `y > 0`; a smaller `sigma` is a heavier tail, and the mean `mu / (sigma - 1)` exists only for `sigma > 1`. | unverified | unverified |
| `PoissonLSS(stabilization: str = 'none')` | `P(\mu)` Poisson location family: `mu` (log link, mean). Matches R `gamboostLSS::PoissonLSS()` (wrapping gamlss.dist `PO`). Requires non-negative integer counts. | unverified | unverified |
| `StudentTLSS(stabilization: str = 'none')` | Student's t family: `mu` (identity link), `sigma` (log link, scale), `df` (log link, degrees of freedom). Matches R `gamboostLSS::StudentTLSS()`. | verified | verified |
| `WeibullLSS(stabilization: str = 'none')` | Weibull AFT family. Takes a right-censored `Surv(time, status)` response, or a plain positive response treated as uncensored. | intentional-deviation | intentional-deviation |
| `ZINBLSS(stabilization: str = 'none')` | Zero-inflated negative-binomial (type I) family: `mu` (log link, mean), `sigma` (log link, dispersion), `nu` (logit link, zero-inflation probability). Matches R `gamboostLSS::ZINBLSS()` (wrapping gamlss.dist `ZINBI`). Requires non-negative integer counts. | unverified | smoke |
| `ZIPoLSS(stabilization: str = 'none')` | Zero-inflated Poisson family: `mu` (log link, Poisson mean), `sigma` (logit link, zero-inflation probability). Matches R `gamboostLSS::ZIPoLSS()` (wrapping gamlss.dist `ZIP`). Requires non-negative integer counts. | verified | verified |
| `ZeroAdjustedGammaLSS(stabilization: str = 'none')` | Zero-adjusted Gamma: `mu` (log, mean of the positive part), `sigma` (log, Gamma dispersion), `nu` (logit, probability of an exact zero). Requires `y >= 0`. Matches `gamlss.dist::ZAGA()`. | unverified | unverified |
| `ZeroAdjustedPoissonLSS(stabilization: str = 'none')` | Zero-adjusted Poisson family: `mu` (log link, Poisson mean), `sigma` (logit link, zero-inflation probability). Matches R `gamboostLSS::ZeroAdjustedPoissonLSS()` (wrapping gamlss.dist `ZAP`). Requires non-negative integer counts. | unverified | unverified |

**`BCCGLSS`**

```python
families = mb.BCCGLSS()  # y: strictly positive
```

**`BCPELSS`**

```python
families = mb.BCPELSS()  # y: strictly positive
```

**`BCTLSS`**

```python
families = mb.BCTLSS()  # y: strictly positive
```

**`BetaLSS`**

```python
families = mb.BetaLSS()  # y: in (0, 1), proportions
```

**`BinomialLSS`**

```python
families = mb.BinomialLSS()  # y in {0, 1} or [0, 1]
```

**`DirichletLSS`**

```python
families = mb.DirichletLSS(K=3)  # cbind(y1, ..., yK): composition rows summing to 1
```

**`GammaLSS`**

```python
families = mb.GammaLSS()  # y: strictly positive
```

**`GaussianKnownVarLSS`**

```python
families = mb.GaussianKnownVarLSS()  # cbind(y, s2): outcome and its known sampling variance
```

**`GaussianLSS`**

```python
families = mb.GaussianLSS()  # y: any real value
```

**`GeometricLSS`**

```python
families = mb.GeometricLSS()  # y: counts
```

**`GumbelLSS`**

```python
families = mb.GumbelLSS()  # y: any real value
```

**`InverseGaussianLSS`**

```python
families = mb.InverseGaussianLSS()  # y: strictly positive
```

**`JSULSS`**

```python
families = mb.JSULSS()  # y: any real value
```

**`LogLogLSS`**

```python
families = mb.LogLogLSS()  # Surv(time, status), log-logistic baseline
```

**`LogNormalLSS`**

```python
families = mb.LogNormalLSS()  # Surv(time, status), log-normal baseline
```

**`LogisticLSS`**

```python
families = mb.LogisticLSS()  # y: any real value
```

**`NBinomialLSS`**

```python
families = mb.NBinomialLSS()  # y: counts
```

**`NegativeBinomialLSS`**

```python
families = mb.NegativeBinomialLSS()  # y: counts
```

**`PIGLSS`**

```python
families = mb.PIGLSS()  # y: counts
```

**`Pareto2LSS`**

```python
families = mb.Pareto2LSS()  # y: strictly positive
```

**`PoissonLSS`**

```python
families = mb.PoissonLSS()  # y: counts
```

**`StudentTLSS`**

```python
families = mb.StudentTLSS()  # y: any real value
```

**`WeibullLSS`**

```python
families = mb.WeibullLSS()  # Surv(time, status), Weibull baseline
```

**`ZINBLSS`**

```python
families = mb.ZINBLSS()  # y: counts with zeros, overdispersed
```

**`ZIPoLSS`**

```python
families = mb.ZIPoLSS()  # y: counts with zeros
```

**`ZeroAdjustedGammaLSS`**

```python
families = mb.ZeroAdjustedGammaLSS()  # y: non-negative, with zeros
```

**`ZeroAdjustedPoissonLSS`**

```python
families = mb.ZeroAdjustedPoissonLSS()  # y: non-negative integers with zeros
```


## Base learners

Specify linear, smooth, spatial, and other effects and their penalties.

| Name | What it is | Python fit | R fit |
|---|---|---|---|
| `Bbs(name, *, knots=20, degree=3, differences=2, df=4, lambda_value=None, center=False, by=None, cyclic=False, boundary_knots=None, constraint=None, deriv=0, bins=None, knot_placement='equidistant')` | Penalized B-spline (P-spline) base-learner (R `bbs`). | verified | verified |
| `Bconstrained(name, *, C, knots=20, degree=3, differences=2, df=4, lambda_value=None, type='quad.prog', lambda2=1000000.0, niter=10, by=None)` | Constrained P-spline base-learner (R `bconstrained`). | smoke | smoke |
| `Bkernel(name, *, bandwidth=1.0, df=4, kernel='gaussian', landmarks=None, center=False, by=None)` | Kernel base-learner (R `bkernel`). | intentional-deviation | intentional-deviation |
| `Bmono(name, *, constraint='increasing', knots=20, degree=3, differences=2, df=4, type='quad.prog', lambda2=1000000.0, niter=10, by=None, center=False)` | Monotone-constrained P-spline learner (R `bmono`). | verified | smoke |
| `Bmrf(name, *, K, df=4, lambda_value=None, center=False, by=None)` | Markov Random Field base-learner (R `bmrf`). | verified | smoke |
| `Bns(name, *, knots=20, degree=3, df=None, center=False, by=None)` | Ridge-penalized B-spline base-learner (mboostrs extension). | verified | smoke |
| `Bols(name, *, intercept=True, df=None, lambda_value=None, by=None, bins=None)` | Ordinary least-squares base-learner (R `bols`). | verified | verified |
| `Brad(names, data=None, *, knots=100, df=4, lambda_value=None, theta=None, center=False, by=None)` | Radial-basis base-learner (R `brad`). | smoke | smoke |
| `Brandom(name, *, df=4, lambda_value=None, center=False, by=None, bins=None)` | Random-effects base-learner (R `brandom`). | verified | verified |
| `Bspatial(name1, name2, *, knots=20, degree=3, differences=2, df=6, lambda_value=None, center=False, by=None, bins=None)` | Tensor-product P-spline base-learner (R `bspatial`). | verified | smoke |
| `Btree(names, *, minbucket=4, minsplit=10, maxdepth=1, by=None, selection='sse', split_method='exact')` | Tree base-learner (R `btree`). Fits decision stumps by default. | verified | smoke |
| `Buser(name, *, X, K, df=4, lambda_value=None, center=False, by=None)` | User-supplied base-learner (R `buser`). | verified | smoke |
| `Hinge(name, *, df=4.0, n_candidates=20, lambda_value=None, center=True, by=None)` | Adaptive hinge base-learner that estimates a piecewise-linear effect. | smoke | unverified |
| `LearnerDesign(term: str, data)` | A fitted design/penalty recipe for `buser` and inspection. |
| `learner_design(term: str, data) -> mboostrs.interactions.LearnerDesign` | Build a reusable design and penalty for one base-learner term. |

**`Bbs`**

```python
mb.gamboost("DEXfat ~ bbs(hipcirc, df=4)", data=df)
```

**`Bconstrained`**

```python
C = ...  # (constraints, p) matrix, p = spline columns
mb.gamboost("DEXfat ~ bconstrained(hipcirc, C=C)",
            data=df, env={"C": C})
```

**`Bkernel`**

```python
mb.gamboost("DEXfat ~ bkernel(hipcirc)", data=df)
```

**`Bmono`**

```python
mb.gamboost("DEXfat ~ bmono(hipcirc)", data=df)
```

**`Bmrf`**

```python
K = ...  # neighbourhood matrix, programmatic
mb.gamboost("rate ~ bmrf(district, K=K)", data=df,
            env={"K": K})
```

**`Bns`**

```python
mb.gamboost("DEXfat ~ bns(hipcirc, df=4)", data=df)
```

**`Bols`**

```python
mb.gamboost("DEXfat ~ bols(waistcirc)", data=df)
```

**`Brad`**

```python
mb.gamboost("DEXfat ~ brad(hipcirc)", data=df)
```

**`Brandom`**

```python
mb.gamboost("rent ~ brandom(district)", data=df)
```

**`Bspatial`**

```python
mb.gamboost("y ~ bspatial(x, z, df=6)", data=df)
```

**`Btree`**

```python
mb.gamboost("DEXfat ~ btree(hipcirc)", data=df)
```

**`Buser`**

```python
X = ...  # (n, p) design; K optional penalty
mb.gamboost("y ~ buser(X, K=K)", data=df,
            env={"X": X, "K": K})
```

**`Hinge`**

```python
mb.gamboost("DEXfat ~ hinge(hipcirc)", data=df)
```


## Cross-validation and folds

Create resampling partitions and select `mstop` from held-out loss.

| Name | What it is |
|---|---|
| `CvReport(folds: str, n_folds: int, grid: tuple, optimum: object, risk_at_optimum: float, score: str, at_boundary: bool, preprocessing: str = 'fixed') -> None` | Summary of a cross-validation run for standard and distributional models. |
| `CvRiskResult(risk_matrix, aggregate, grid, optimal_mstop, fold_type, provenance=None, *, fold_fingerprint='', data_fingerprint='', folds=None, held_out_fraction=None, design=None, preprocessing='fixed', warn_boundary=True, comparison_identity=None)` | Cross-validated risk along the boosting path. |
| `EvaluationResult(score: float, fold_scores: numpy.ndarray, formula: str, response: str, family: str, score_name: str, score_parameter: float \| None, prediction_scale: str, loss_units: str, aggregation: str, fold_scheme: str, evaluation_task: str, independent_unit: str, n_independent_units: int, seed: int, tuning: str, data_fingerprint: str, fingerprint_scope: str, n_observations: int, n_evaluations: int, fit_count: int, excluded_record_ids: tuple, records: tuple[dict, ...], folds: tuple[dict, ...], warnings: tuple[str, ...] = (), class_labels: tuple \| None = None, build_id: str = '', response_transform: str = 'identity', weights_fingerprint: str = '', data_identity: str = '') -> None` | Held-out predictions, scoring semantics and provenance for one specification. |
| `FoldScheme(weights: numpy.ndarray, held_out: numpy.ndarray, scheme: str, seed: int \| None = None, groups: numpy.ndarray \| None = None, strata: numpy.ndarray \| None = None, ordered: bool = False, judge: numpy.ndarray \| None = None, _note: str = '', base_weights: numpy.ndarray \| None = None, repeats: int = 1, known_groups: numpy.ndarray \| None = None, times: numpy.ndarray \| None = None, embargo: float \| numpy.timedelta64 \| None = None) -> None` | An `(n, B)` in-bag weight matrix, and what it means. |
| `LssCvRiskResult(risk_matrix: numpy.ndarray, aggregate: numpy.ndarray, grid: list[list[int]], optimal_mstop: list[int], fold_type: str = 'custom', provenance: object = None, param_names: tuple = (), data_identity: str = '') -> None` | Cross-validated LSS risk over a per-parameter mstop grid. |
| `MstopTuning(mstop: int, risk: tuple, largest: int, first_largest: int, at_boundary: bool, fits: int, _fit: object = None, _fit_at_mstop: bool = False, score: str = 'family', score_parameter: float \| None = None, full_data_fits: int = 2, fold_scores: tuple = (), grid_ends: tuple = (), patience: int \| None = None) -> None` | A cross-validated step count, and whether the grid grew to find it. |
| `blocked_folds(n, B=5, *, weights=None) -> mboostrs.folds.FoldScheme` | Split rows into contiguous held-out blocks. |
| `cv_weights(n, type='bootstrap', B=25, prob=0.5, seed=42, weights=None)` | Generate cross-validation fold weights. |
| `cvrisk_gamboostLSS(formula, data, families=None, family=None, control=None, folds=None, grid=None, bins=None, env=None, weight_policy='upstream', nu=None, stabilization=None, weights=None, na_action='error', _search=None, response_scale='auto')` | Cross-validated risk for a mixed structured `gamboostLSS` model. |
| `cvrisk_glmboostLSS(formula, data, families=None, family=None, control=None, folds=None, grid=None, weight_policy='upstream', nu=None, stabilization=None, weights=None, response_scale='auto')` | Cross-validated risk for a linear GAMLSS over a grid of per-parameter mstop combinations (R's `cvrisk.mboostLSS` for `glmboostLSS`). |
| `diagonal_grid(max_mstop, n_params, length_out=10, log=True)` | A grid whose points give every distribution parameter the same budget. |
| `evaluate_specification(rhs, data, response, **kwargs)` | Evaluate one specification and retain its held-out evidence. |
| `grouped_folds(groups, B=10, *, seed=42, weights=None, repeats=1) -> mboostrs.folds.FoldScheme` | K-fold splits that keep each group entirely on one side. |
| `known_group_folds(groups, B=10, *, seed=42, weights=None) -> mboostrs.folds.FoldScheme` | K-fold splits in which every held-out row has a group-mate in training. |
| `loo_folds(n, *, weights=None) -> mboostrs.folds.FoldScheme` | Leave-one-out: fold `i` holds out row `i` alone (`n` folds). |
| `make_grid(max, length_out=10, min=None, log=True, dense_mu_grid=True)` | Build a per-parameter `mstop` grid for distributional cross-validation. |
| `make_grid_lss(max_mstop, length_out=10, log=True)` | Per-parameter 1-D mstop grid, mirroring gamboostLSS::make.grid's marginal sequence (the Cartesian product over parameters forms the full search grid). |
| `repeated_kfold_folds(n, k=10, repeats=5, *, seed=42, weights=None) -> mboostrs.folds.FoldScheme` | `repeats` independent k-fold partitions, side by side. |
| `rolling_folds(n, B=5, *, horizon=None, initial=None, expanding=True, weights=None, time=None, embargo=None) -> mboostrs.folds.FoldScheme` | Rolling-origin splits: train on the past, judge on what comes next. |
| `stratified_folds(y, B=10, *, seed=42, weights=None) -> mboostrs.folds.FoldScheme` | K-fold splits that preserve the class proportions of `y`. |
| `tune_mstop(formula, data, family=None, *, folds=5, start=100, nu=None, extensions=None, max_mstop=10000, seed=0, step='fixed', score='family', score_parameter=None, unseen_levels='average', fold_local=False, patience=None, fit_budget=None, extrapolation='warn', flat_tolerance=0.01, **fit_kwargs)` | Cross-validate the step count, doubling the grid while the risk still falls. |

**`CvReport`**

```python
report = fit.cvrisk(folds=folds).report()
report.optimum, report.risk_at_optimum, report.at_boundary
```

**`EvaluationResult`**

```python
evaluation.fold_frame(output="polars")
evaluation.describe()
```

**`FoldScheme`**

```python
scheme = mb.rolling_folds(120, B=4, initial=60, horizon=12)
fit.cvrisk(folds=scheme).held_out_fraction
```

**`blocked_folds`**

```python
folds = mb.blocked_folds(n=120, B=5)
fit.cvrisk(folds=folds)
```

**`cv_weights`**

```python
folds = mb.cv_weights(n=71, type="kfold", B=5, seed=42)
curve = fit.cvrisk(folds=folds)
curve.optimal_mstop
```

**`evaluate_specification`**

```python
evaluation = mb.evaluate_specification(
    "bols(x1) + bbs(x2, df=3)",
    mb.datasets.heteroscedastic_lss(), "y", folds=5, score="mse",
)
```

**`grouped_folds`**

```python
folds = mb.grouped_folds(groups, B=4)
fit.cvrisk(folds=folds)
```

**`known_group_folds`**

```python
folds = mb.known_group_folds(groups, B=4)
mb.evaluate_specification("bols(x) + brandom(g)",
                          data, "y", folds=folds)
```

**`loo_folds`**

```python
folds = mb.loo_folds(n=60)
fit.cvrisk(folds=folds)
```

**`repeated_kfold_folds`**

```python
folds = mb.repeated_kfold_folds(n=120, k=5,
                        repeats=10, seed=1)
fit.cvrisk(folds=folds)
```

**`rolling_folds`**

```python
folds = mb.rolling_folds(n=120, B=4, initial=60,
                 horizon=12)
fit.cvrisk(folds=folds)
```

**`stratified_folds`**

```python
folds = mb.stratified_folds(y_binary, B=5)
fit.cvrisk(folds=folds)
```

**`tune_mstop`**

```python
tuning = mb.tune_mstop("DEXfat ~ bbs(hipcirc, df=4)",
                       data=mb.datasets.bodyfat(), folds=5)
tuning.mstop
```


## Stability selection

Select terms by resampling, compare candidate effects, and refit selected specifications.

| Name | What it is |
|---|---|
| `Deselection(kept: tuple, dropped: tuple, shares: dict, threshold: float, formula: str \| dict, model: object) -> None` | The terms `deselect` kept and dropped, and the refitted model. |
| `EffectContrast(estimate: float, se: float, lower: float, upper: float, level: float, scale: str = 'link') -> None` | The average change in a linear predictor when covariates move, with its interval. |
| `EquivalenceSets(groups: tuple, overlap: dict, threshold: float, measured: tuple = (), scope: str = 'Predictive substitution in the rows and resamples used. A group names learners that carry the same information there; it does not identify which member is causal, and it is not a causal claim.') -> None` | Groups of learners whose design spaces overlap at or above a threshold. |
| `GamRefit(terms: tuple, edf: dict, smoothing: dict, sigma: float, conditional_on_selection: bool, link: str = 'identity', reml_score: float = nan, convergence: dict = <factory>, theta: float \| None = None, held: tuple = (), _blocks: tuple = (), _core: object = None, _held_model: object = None, _held_learners: tuple = (), _held_scale: float = 1.0, _impute: object = None, _training_data: object = None, _ranges: dict = <factory>, _weights: object = None) -> None` | A jointly penalized refit of a set of terms, with Bayesian intervals. |
| `GamlssRefit(family: str, terms: dict, edf: dict, smoothing: dict, reml_score: float, convergence: dict, conditional_on_selection: bool, _blocks: dict = <factory>, _core: object = None, _replay: object = None, _training_data: object = None, _ranges: dict = <factory>, intercepts: dict = <factory>, intercept_only: tuple = ()) -> None` | A penalized refit of a distributional model's terms, all parameters estimated jointly, with REML smoothing parameters. |
| `LssStabSelResult(phat: numpy.ndarray, selected: list[str], max_: numpy.ndarray, cutoff: float, q: int, PFER: float, learner_names: list[str], sampling_type: str = 'SS', assumption: str = 'unimodal', B: int = 0, n_fits: int = 0, provenance: object = None) -> None` | Stability-selection result for combined LSS learner/parameter effects. |
| `Probing(selected: tuple, stop: int \| None, first_shadow: str \| None, formula: str, model: object) -> None` | Result of `probe`: selected terms, the first shadow, and its entry iteration. |
| `SelectionReport(equivalence: mboostrs.equivalence.EquivalenceSets, substitution: mboostrs.equivalence.SubstitutionSets \| None, dropped_with_substitute: tuple, dropped_alone: tuple, scope: str = 'Predictive substitution in the rows and resamples used. A group names learners that carry the same information there; it does not identify which member is causal, and it is not a causal claim.') -> None` | Equivalence sets, substitution sets, and the deselected learners they explain. |
| `SplitRefitResult(formula: object, family: str, select: str, repeats: int, fraction: float, level: float, seed: int, scale: str, terms: tuple, contrasts: tuple, splits: tuple = ()) -> None` | The output of `split_refit`: the aggregate, and the per-repeat records. |
| `StabSelResult(phat: numpy.ndarray, selected: list[int], max_: numpy.ndarray, cutoff: float, q: int, PFER: float, p: int, B: int, sampling_type: str, assumption: str, learner_names: list[str], n_fits: int = 0, provenance: object = None, data_fingerprint: str = '', fit_selected: tuple = ()) -> None` | Stability-selection frequencies, selected learners, and error bound. |
| `SubstitutionSets(groups: tuple, pairs: tuple, subsamples: int, min_share: float, max_joint: float, min_overlap: float \| None = None, scope: str = 'Predictive substitution in the rows and resamples used. A group names learners that carry the same information there; it does not identify which member is causal, and it is not a causal claim.') -> None` | Substitute pairs and the groups they form, from a stability selection run. |
| `deselect(model, threshold=0.01, *, folds=5, seed=0, mstop=None)` | Drop base learners with a small share of the risk reduction, then refit. |
| `equivalence_sets(model, *, threshold=0.9)` | Learners whose design spaces overlap at or above `threshold`. |
| `gam_refit(model, terms=None, *, which='selected', unsupported='error')` | Refit a boosted model's terms as one penalized GAM with REML smoothing. |
| `gamlss_refit(model, terms=None, *, which='selected', factor_ridge=False)` | Refit a distributional model's terms as one penalized likelihood with REML smoothing parameters. |
| `probe(formula, data=None, *, family=None, control=None, seed=0, folds=5, **fit_kwargs)` | Select terms by the first entry of a permuted shadow, then refit them. |
| `selection_report(model, *, deselection=None, stability=None, equivalence_threshold=0.9, min_share=0.1, max_joint=0.2, min_overlap=0.5)` | Equivalence sets, substitution sets and deselection, read together. |
| `split_refit(formula, data, *, family=None, select='deselect', repeats=1, fraction=0.5, level=0.9, seed=0, contrasts=None)` | Choose terms on one random half of the rows and estimate them by REML on the other half, optionally over several random splits. |
| `stabsel(model, cutoff=None, q=None, PFER=None, B=None, folds=None, assumption='unimodal', sampling_type='SS', grid=None, verbose=True, eval=True, n_jobs=-1, fit_budget=None)` | Stability selection for mboostrs models. |
| `stabsel_gamboostLSS(formula, data, families=None, family=None, control=None, folds=None, q=None, PFER=None, cutoff=None, B=50, sampling_type='SS', assumption='unimodal', seed=42, weights=None, bins=None, env=None, weight_policy='upstream', nu=None, stabilization=None, fit_budget=None, na_action='error', response_scale='auto')` | Stability selection for a mixed structured `gamboostLSS` model. |
| `stabsel_glmboostLSS(formula, data, families=None, family=None, control=None, folds=None, q=None, PFER=None, cutoff=None, B=50, sampling_type='SS', assumption='unimodal', seed=42, weights=None, weight_policy='upstream', nu=None, stabilization=None, fit_budget=None, response_scale='auto')` | Stability selection for a linear GAMLSS (R's stabsel.mboostLSS). |
| `substitution_sets(stability, *, min_share=0.1, max_joint=0.2, equivalence=None, min_overlap=0.5)` | Learners that stability selection picks in turn rather than together. |

**`Deselection`**

```python
tuned = mb.tune_mstop("DEXfat ~ .", data=mb.datasets.bodyfat()).model
mb.deselect(tuned).dropped
```

**`EffectContrast`**

```python
body = mb.datasets.bodyfat()
fit = mb.gam("DEXfat ~ bbs(hipcirc, knots=6)", data=body)
change = fit.contrast("hipcirc", 90.0, 110.0, data=body)
(change.estimate, change.lower, change.upper)
```

**`EquivalenceSets`**

```python
tuned = mb.tune_mstop("DEXfat ~ .", data=mb.datasets.bodyfat()).model
mb.equivalence_sets(tuned).overlap
```

**`GamRefit`**

```python
body = mb.datasets.bodyfat()
tuned = mb.tune_mstop("DEXfat ~ bbs(hipcirc) + bbs(waistcirc)", data=body).model
estimate, lower, upper = mb.gam_refit(tuned).effects(body)["bbs(hipcirc)"]
(lower[:3], upper[:3])
```

**`GamlssRefit`**

```python
data = mb.datasets.heteroscedastic_lss()
fit = mb.gamboostLSS({"mu": "y ~ bbs(x1)", "sigma": "y ~ bbs(x2)"},
                     data=data, families=mb.GaussianLSS())
mb.gamlss_refit(fit).predict(data, parameter="sigma", type="response")[:3]
```

**`Probing`**

```python
probing = mb.probe("DEXfat ~ .", data=mb.datasets.bodyfat())
(probing.stop, probing.first_shadow)
```

**`SelectionReport`**

```python
tuned = mb.tune_mstop("DEXfat ~ .", data=mb.datasets.bodyfat()).model
mb.selection_report(tuned, deselection=mb.deselect(tuned)).equivalence.groups
```

**`SplitRefitResult`**

```python
data = mb.datasets.bodyfat()
result = mb.split_refit("DEXfat ~ bbs(hipcirc) + bbs(waistcirc)", data,
                        seed=0, contrasts=[("hipcirc", 90.0, 110.0)])
result.to_dict()["contrasts"]
```

**`SubstitutionSets`**

```python
tuned = mb.tune_mstop("DEXfat ~ .", data=mb.datasets.bodyfat()).model
stab = mb.stabsel(tuned, cutoff=0.6, PFER=2.0, B=50, verbose=False)
mb.substitution_sets(stab, equivalence=mb.equivalence_sets(tuned)).pairs
```

**`deselect`**

```python
tuned = mb.tune_mstop("DEXfat ~ .", data=mb.datasets.bodyfat()).model
mb.deselect(tuned).kept
```

**`equivalence_sets`**

```python
tuned = mb.tune_mstop("DEXfat ~ .", data=mb.datasets.bodyfat()).model
mb.equivalence_sets(tuned, threshold=0.9).groups
```

**`gam_refit`**

```python
tuned = mb.tune_mstop("DEXfat ~ bbs(hipcirc) + bbs(waistcirc)",
                      data=mb.datasets.bodyfat()).model
mb.gam_refit(tuned).edf
```

**`gamlss_refit`**

```python
fit = mb.gamboostLSS({"mu": "y ~ bbs(x1)", "sigma": "y ~ bbs(x2)"},
                     data=mb.datasets.heteroscedastic_lss(),
                     families=mb.GaussianLSS())
mb.gamlss_refit(fit).edf
```

**`probe`**

```python
mb.probe("DEXfat ~ .", data=mb.datasets.bodyfat()).selected
```

**`selection_report`**

```python
tuned = mb.tune_mstop("DEXfat ~ .", data=mb.datasets.bodyfat()).model
mb.selection_report(tuned, deselection=mb.deselect(tuned)).dropped_alone
```

**`split_refit`**

```python
data = mb.datasets.bodyfat()
result = mb.split_refit("DEXfat ~ bbs(hipcirc) + bbs(waistcirc)", data,
                        repeats=5, seed=0,
                        contrasts=[("hipcirc", 90.0, 110.0)])
result.summary()
```

**`stabsel`**

```python
sel = fit.stabsel(cutoff=0.8, q=2, PFER=1.5, B=50)
sel.selected
```

**`substitution_sets`**

```python
tuned = mb.tune_mstop("DEXfat ~ .", data=mb.datasets.bodyfat()).model
stab = mb.stabsel(tuned, cutoff=0.6, PFER=2.0, B=50, verbose=False)
mb.substitution_sets(stab, equivalence=mb.equivalence_sets(tuned)).groups
```


## Diagnostics

Inspect residuals, calibration, prediction intervals, and model diagnostics.

| Name | What it is |
|---|---|
| `ConformalInterval(lower: numpy.ndarray, upper: numpy.ndarray, predicted: numpy.ndarray, level: float, method: str, n_calibration: int, quantile: float \| None = None, guaranteed_coverage: float \| None = None, guarantee: str = '', assumptions: tuple[str, ...] = (), n_training: int \| None = None, n_folds: int \| None = None, mstop_source: str = '', calibration_source: str = '', n_set_aside: int = 0, fitted_mean: numpy.ndarray \| None = None, score: str \| None = None, requested_score: str \| None = None, support: str = 'real', reference_distribution: str \| None = None, zero_probability: float \| None = None, reference_dispersion: float \| None = None, randomization_seed: int \| None = None, reliability: dict \| None = None) -> None` | Prediction intervals with their calibration evidence attached. |
| `ConformalReliabilityWarning` | Warning emitted when conformal reliability diagnostics detect risk to coverage. |
| `DistributionEvaluationResult(family: str, response: str, data_role: str, data_fingerprint: str, observed: numpy.ndarray, weights: numpy.ndarray, parameters: dict[str, numpy.ndarray], density_or_mass: numpy.ndarray, negative_log_score: numpy.ndarray, deviance: numpy.ndarray, pit: numpy.ndarray, crps: numpy.ndarray \| None, metrics: dict, intervals: tuple[dict, ...], exceedances: tuple[dict, ...], assumptions: tuple[str, ...]) -> None` | Row-level distribution scores and aggregate predictive diagnostics. |
| `InfluenceFinding(check: str, flagged: bool, pattern: str, statistic: str, value: float, units: str, p_value: float \| None, reference: str, false_alarm_rate: float, false_alarm_draws: int, assumption: str, alternatives: tuple[str, ...], revision: str, held_out_check: str, evidence: dict[str, typing.Any] = <factory>, rows: tuple[int, ...] = (), cooks: numpy.ndarray \| None = None, loo_residual: numpy.ndarray \| None = None, studentized: numpy.ndarray \| None = None, leverage: numpy.ndarray \| None = None, refits: tuple[dict, ...] = ()) -> None` | The influence check, with the per-row measures and exact deletion refits. |
| `RearrangedQuantiles(kind: str, levels: tuple, predictions: Any, unsorted: Any, crossing_rows: int, crossing_share: float, pair_crossings: Any, max_crossing: float, n: int) -> None` | Sorted predictions at several levels and the crossings found before sorting. |
| `RevisionFinding(check: str, flagged: bool, pattern: str, statistic: str, value: float, units: str, p_value: float \| None, reference: str, false_alarm_rate: float, false_alarm_draws: int, assumption: str, alternatives: tuple[str, ...], revision: str, held_out_check: str, evidence: dict[str, typing.Any] = <factory>) -> None` | One residual check: what it found, the evidence, and the revision. |
| `RevisionReport(findings: tuple[mboostrs.residual_checks.RevisionFinding, ...]) -> None` | The three checks run together, in the order cluster, time, influence. |
| `SameFamilyGap(available: bool, family: str, mstop: object, mstop_source: str, boost_loglik: float \| None, comparison_loglik: float \| None, gap: float \| None, material: bool, comparison: str, message: str, tol: float = 1.0, n_obs: int \| None = None) -> None` | Log-likelihood gap between a boost and the same-family GLM. |
| `ScaleAbsorption(verdict: str, rule: str, absorbed: dict, scale: dict, oof: dict, residual_structure: dict, specifications: dict, folds: int, seed: int, notes: tuple) -> None` | Result of `scale_absorption`. |
| `SurvivalEvaluationResult(family: str, response: tuple, data_role: str, data_fingerprint: str, horizons: tuple, n_observations: int, total_weight: float, censoring_floor: float, on_low_censoring: str, time: numpy.ndarray, event: numpy.ndarray, weights: numpy.ndarray, linear_predictor: numpy.ndarray, survival: numpy.ndarray, horizon_rows: tuple[dict, ...], assumptions: tuple[str, ...]) -> None` | Fixed-horizon Brier score and time-dependent AUC of a CoxPH model. |
| `acf_residuals(model, max_lag=None, powers=(1, 2, 3, 4), kind=None, newdata=None, mstop=None, output='auto')` | Compute residual autocorrelation and autocorrelation of selected powers. |
| `calibration_frame(model, xvar=None, bins=10, newdata=None, mstop=None, output='auto')` | Compare observed and fitted means within bins. |
| `centile_fan_data(model, xvar, probs=(0.025, 0.1, 0.25, 0.5, 0.75, 0.9, 0.975), split_by=None, newdata=None, mstop=None, output='auto')` | Observed responses and fitted centiles ordered by one numeric covariate. |
| `cluster_residual_check(model, cluster, *, alpha: float = 0.05, permutations: int = 999, draws: int = 200, seed: int = 0) -> mboostrs.residual_checks.RevisionFinding` | Measure the share of response-residual variance between clusters. |
| `coef_all(model, output='auto')` | Return one coefficient row per learner, boosting step and component. |
| `conformal_interval(model, newdata=None, *, level=0.9, method='split', seed=None, calib_fraction=0.5, n_folds=10, mstop=None, calibration=None, extrapolation='warn', score='auto', count_reference='zero_inflated_poisson', unseen_levels=None)` | Distribution-free prediction intervals for a fitted standard model. |
| `coverage(observed, lower, upper)` | Compute empirical coverage of prediction intervals. |
| `density_overlay_data(model, newdata=None, mstop=None, grid=None, n_grid=200, weights=None, output='auto')` | Observed and average fitted response distributions on one grid. |
| `deviance_increment(model, newdata=None)` | Per-observation deviance contributions. |
| `deviance_profile(model, xvar=None, bins=10, newdata=None, mstop=None, weights=None, output='auto')` | Bin joint deviance contributions along a numeric covariate. |
| `evaluate_distribution(model, data, response=None, *, data_role, weights=None, levels=(0.5, 0.8, 0.95), thresholds=(), random_state=0, mstop=None, unseen_levels='error')` | Evaluate a fitted scalar-response LSS distribution on supplied data. |
| `evaluate_survival(model, data, *, horizons, data_role, weights=None, censoring_floor=0.05, on_low_censoring='error')` | Brier score and time-dependent AUC of a CoxPH model at fixed horizons. |
| `heteroscedasticity(model, data=None, *, columns=None, transforms=None) -> dict` | Measure how residual spread changes across fitted values. |
| `imputation_diagnostics(model) -> dict` | Report the R-squared of each model-based predictor fill, by column. |
| `influence_check(model, *, alpha: float = 0.05, refit_top: int = 5, draws: int = 200, seed: int = 0) -> mboostrs.residual_checks.InfluenceFinding` | Use leverage and deletion refits to assess changes associated with individual rows. |
| `interval_metrics(observed, lower, upper, *, level=0.95, weights=None, retain_scores=False)` | Weighted coverage, width, and proper central-interval score. |
| `ipc_weights(time, status, maxweight: float = 5.0) -> numpy.ndarray` | Inverse probability of censoring weights (mboost `IPCweights`). |
| `outlier_frame(model, newdata=None, mstop=None, output='auto')` | Observations ranked by absolute residual, largest first. |
| `pit(model, newdata=None, mstop=None, randomized=True, random_state=None)` | Probability integral transform for LSS models with CDF support. |
| `predict_all(model, newdata=None, type='link', mstop=None, include_response=True, output='auto', extrapolation='warn')` | Return a table of fitted values (and response for in-sample). |
| `q_stats(model, xvar=None, n_intervals=10, newdata=None, mstop=None, randomized=True, random_state=None, output='auto')` | Binned quantile-residual moment summaries. |
| `quantile_residuals(model, newdata=None, mstop=None, randomized=True, random_state=None)` | Randomized quantile residuals. |
| `rearrange_quantiles(models, newdata=None, *, levels=None, output='auto')` | Predict several quantile or expectile fits and sort each row across levels. |
| `residual_frame(model, kind='response', newdata=None, mstop=None, randomized=True, random_state=None, output='auto')` | Diagnostic residual table for any model. |
| `revision_checks(model, *, cluster=None, time=None, alpha: float = 0.05, permutations: int = 999, draws: int = 200, seed: int = 0, refit_top: int = 5) -> mboostrs.residual_checks.RevisionReport` | Run the cluster, time and influence checks and report their revisions. |
| `same_family_gap(model, cv=None, *, comparison_loglik=None, tol=1.0)` | Compare a linear boost at its cross-validated step with the same-family GLM. |
| `scale_absorption(model, *, folds=5, seed=0, flexible=None)` | Test whether a fitted LSS scale effect survives a more flexible mean. |
| `time_residual_check(model, time, *, max_lag: int \| None = None, alpha: float = 0.05, permutations: int = 999, draws: int = 200, seed: int = 0) -> mboostrs.residual_checks.RevisionFinding` | Test whether residuals are correlated across ordered time points. |
| `validate(model, newdata, *, offset=None, output='auto')` | The issues that `predict(newdata)` would meet, before it runs. |
| `worm_data(model, xvar=None, n_intervals=4, newdata=None, mstop=None, randomized=True, random_state=None, output='auto')` | Return worm-plot deviation data. |

**`ConformalReliabilityWarning`**

```python
import warnings
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", mb.ConformalReliabilityWarning)
    ci = mb.conformal_interval(model, newdata=new_df, method="split")
ci.reliability["status"]
```

**`DistributionEvaluationResult`**

```python
scores = evaluation.to_frame(output="polars")
evaluation.describe()
```

**`InfluenceFinding`**

```python
finding = mb.influence_check(fit)
finding.cooks, finding.rows
```

**`RearrangedQuantiles`**

```python
r = mb.rearrange_quantiles([fit_q10, fit_q50, fit_q90], rows)
r.pair_crossings, r.max_crossing
```

**`RevisionFinding`**

```python
finding = mb.cluster_residual_check(fit, "site")
print(finding.summary)
```

**`RevisionReport`**

```python
report = mb.revision_checks(fit, cluster="site")
report.flagged
```

**`ScaleAbsorption`**

```python
result = mb.scale_absorption(fit, folds=5, seed=0)
result.to_dict()["absorbed"]
```

**`SurvivalEvaluationResult`**

```python
evaluation.survival_frame(output="polars")
evaluation.describe()
```

**`centile_fan_data`**

```python
table = mb.centile_fan_data(lss_fit, xvar="x")
```

**`cluster_residual_check`**

```python
finding = mb.cluster_residual_check(fit, "site")
finding.flagged, finding.revision
```

**`conformal_interval`**

```python
interval = mb.conformal_interval(fit, level=0.9)
interval.lower[:3]
```

**`coverage`**

```python
mb.coverage(observed, lower, upper)
```

**`density_overlay_data`**

```python
table = mb.density_overlay_data(lss_fit)
```

**`deviance_profile`**

```python
table = mb.deviance_profile(lss_fit, xvar="x")
```

**`evaluate_distribution`**

```python
evaluation = mb.evaluate_distribution(
    lss_fit, test_data, response="y", data_role="external",
    levels=(0.8, 0.95), thresholds=(0,),
)
```

**`evaluate_survival`**

```python
evaluation = mb.evaluate_survival(
    cox_fit, test_data, horizons=[1.0, 2.0], data_role="external",
)
evaluation.horizon_frame(output="polars")
```

**`heteroscedasticity`**

```python
report = mb.heteroscedasticity(fit)
report["spread_ratio"]
```

**`imputation_diagnostics`**

```python
fit = mb.gamboost("y ~ bbs(x) + bbs(z)", data,
                  na_action="impute_model")
mb.imputation_diagnostics(fit)  # {"x": R-squared}

```

**`influence_check`**

```python
finding = mb.influence_check(fit)
finding.rows, finding.refits
```

**`interval_metrics`**

```python
metrics = mb.interval_metrics(y, interval.lower, interval.upper,
                              level=0.9)
```

**`pit`**

```python
mb.pit(fit)
```

**`q_stats`**

```python
mb.q_stats(fit)
```

**`quantile_residuals`**

```python
mb.quantile_residuals(fit)
```

**`rearrange_quantiles`**

```python
r = mb.rearrange_quantiles([fit_q10, fit_q50, fit_q90], rows)
r.crossing_share, r.predictions
```

**`revision_checks`**

```python
report = mb.revision_checks(fit, cluster="site", time="year")
report.revisions
```

**`same_family_gap`**

```python
fit = mb.glmboost("y ~ x",
    data={"y": [1.0, 2.0, 1.4, 2.2], "x": [0.0, 1.0, 0.2, 0.9]},
    control=mb.boost_control(30, 0.2))
gap = mb.same_family_gap(fit)
gap.mstop, gap.gap, gap.material
```

**`scale_absorption`**

```python
result = mb.scale_absorption(fit, folds=5, seed=0)
result.verdict
```

**`time_residual_check`**

```python
finding = mb.time_residual_check(fit, "year")
finding.statistic, finding.p_value
```

**`validate`**

```python
issues = mb.validate(model, test, output="records")
refused = [issue for issue in issues if issue["severity"] == "error"]
```


## Choosing a specification

Compare candidate families and terms using the specified folds and scoring rules.

| Name | What it is |
|---|---|
| `CapacitySelection(selected: str, formula: str, rhs: str, tolerance: float, rungs: tuple[dict[str, typing.Any], ...], screened_pairs: tuple[dict[str, typing.Any], ...], model: object, selected_estimator: str = 'boosting') -> None` | The capacity ladder `select_capacity` measured, and what it kept. |
| `ColumnKind(name: str, levels: int, kind: str, term: str, dtype: str = 'numeric', level_names: tuple \| None = None, ordered: bool = True, n_missing: int = 0, unit: str \| None = None, reason: str = '', alternatives: tuple[tuple[str, str], ...] = ()) -> None` | What a column looks like, the learner that follows, and what else it could be. |
| `CostBudget(seconds: float \| None = None, fits: int \| None = None, memory_mb: float \| None = None) -> None` | Limits on a procedure's fit count, run time and memory. `None` means no limit on that field. |
| `DeclarationCheck(declaration: str, kind: str, variable: str, test: str \| None, verdict: str, agreement: str, evidence: dict = <factory>, reason: str = '') -> None` | The test of one declaration against the fit. |
| `DiscontinuityHit(kind: str, columns: tuple, direction_deg: float, threshold: float, threshold_x: float, boundary_intercept: float, boundary_slope: float, line: str, n_above: int, jump_fit: float, t_fit: float, jump: float, se: float, z: float, gain: float, type_z: float, type_ratio: float, type_gap_t: float) -> None` | One evaluated boundary from `screen_discontinuities`. |
| `DiscoveryResult(model: Any, refit: Any, formula: str, spread_formula: str \| None, family: str, mstop: int, folds: int, seed: int, level: float, mean_effects: list[dict], interactions: list[dict], spread_drivers: list[dict], predictions: polars.dataframe.frame.DataFrame \| None, extrapolation: polars.dataframe.frame.DataFrame \| None, warnings: list[str], log: list[dict], provenance: dict, _frame: polars.dataframe.frame.DataFrame, _variables: frozenset = frozenset(), constraints: dict = <factory>, _dropped_refit: Any = None, _dropped_variables: frozenset = frozenset(), interval: Any = None, stage_seconds: dict = <factory>) -> None` | What `discover` found, with the decisions that produced it. |
| `Finding(finding: str, effect_size: float, uncertainty: float, basis: str = 'measured', evidence: dict = <factory>, next_action: str = '') -> None` | One thing the discovery pipeline found, and what to do about it. |
| `FitCache(*, build=None, enabled=True)` | Memo for evaluated specifications within a Python process. |
| `LLMJudge(*, llm='nvidia/nemotron-3-ultra-550b-a55b', base_url=None, api_key=None, client=None, temperature=0.0, max_tokens=2000, timeout=180, retain_text=False, max_calls=None, max_total_tokens=None)` | A judge backed by any OpenAI-compatible chat endpoint. |
| `LabelEvidence(column: str, levels: int, signal_ratio: float, signal_z: float, adjacency_ratio: float, labels: bool, order_p: float = nan) -> None` | Whether a whole-number column's values act as labels or as a measurement. |
| `ModelerResult(spec: str, family: object, mstop: int, groups: tuple, transcript: mboostrs.modeler.Transcript, ledger: mboostrs.modeler.AssumptionLedger, refusals: list, fits: int, causal_result: object = None, trap_report: object = None, judgment_calls: int = 0, model: object = None, stop_reason: str = 'round_limit', locked_columns: tuple = (), budget_report: dict \| None = None, action_evidence: tuple = ()) -> None` | Model and specification selected by the loop, with the decision record. |
| `OrderedCategoryWarning` | An ordered categorical predictor was fitted as unordered groups. |
| `PairedTest(rung: str, baseline_rhs: str, candidate_rhs: str, baseline: float, candidate: float, gain: float, se: float, t: float, threshold_se: float, passed: bool, worse_folds: int, n_folds: int, adopted: bool, reason: str) -> None` | One rung's decision: the paired fold comparison behind an adoption. |
| `PolicyResult(spec: str, family: object, mstop: int, profile: str, model: object, trail: tuple, spent: mboostrs.policy.Spent, cache: mboostrs.policy.FitCache, score: float \| None, threshold_se: float, complexity: str, stopped_early: str \| None, tuning: object = None, response: str = '', metric: str = 'mse', budget_report: dict \| None = None, action_evidence: tuple = ()) -> None` | Specification selected by the policy, with evidence for each step. |
| `ReferenceGapResult(gap: float, gap_se: float, material: bool, relative_gain: float, reference_loss: float, model_loss: float, model_formula: str, reference_formula: str, model_mstop: int, reference_mstop: int, model_at_cap: bool, reference_at_cap: bool, reference_in_sample_loss: float, candidates: polars.dataframe.frame.DataFrame, closed_share: float \| None, combined_formula: str \| None, combined_gain: float \| None, combined_gain_se: float \| None, folds: int, family: str, notes: list[str] = <factory>, data: polars.dataframe.frame.DataFrame \| None = None, fold_losses: dict = <factory>, reference_columns: list = <factory>, reference_extra_columns: list = <factory>, screen_residual: str = 'none') -> None` | Result of comparing a specification with a tree reference, including candidate revisions. |
| `ScreenedPair(left: str, right: str, removed: float) -> None` | One candidate interaction and the mean squared error it removed. |
| `SearchEvidence(action: str, status: str, evidence_id: str, state_id: str, fit_charges: int, seconds: float, summary: dict, value: object = None) -> None` | One action's result and its position in the search's audit trail. |
| `SearchSession(data, response, *, family=None, score='mse', folds=5, seed=0, weights=None, groups=(), budget=None, na_action='error', unseen_levels='average', score_parameter=None, nu=0.1, mstop=100, locks=(), causal=None, forbidden_pairs=(), mandatory_terms=(), heredity='none')` | A persistent training-only search whose next diagnostic is caller-chosen. |
| `SpecificationCurve(source: str, response: str, score: str, score_parameter: object, mstop: int, nu: float, fold_scheme: str, partitions: int, folds_per_partition: int, selected: str, records: tuple, joint_test: dict, post_hoc: bool, label: str, fits: int, estimated_seconds: float, seconds: float, budget: int \| None, excluded_rows: int, notes: tuple, output: str = 'auto') -> None` | The candidate specifications of a search, scored on one fold partition. |
| `SpecificationJudgment(drop: tuple[str, ...] = (), add: tuple[str, ...] = (), lock: tuple[str, ...] = (), replace: dict = <factory>, monotone: dict = <factory>, flags: tuple = (), assumptions: tuple = (), reasoning: str = '', estimand_note: str = '', evidence_id: str \| None = None) -> None` | Validated editable proposal; predictive evidence decides adoption. |
| `Spent(seconds: float = 0.0, fits: int = 0, memory_mb: float \| None = None, budget: mboostrs.policy.CostBudget \| None = None, stages: list = <factory>) -> None` | Fit, time and memory charges recorded during a procedure. |
| `Suggestion(action: str, term: str, expected_gain: float, reason: str, basis: str = 'measured', specification: str = '', evidence: dict = <factory>) -> None` | One change worth trying, and the evidence for it. |
| `TermReport(formula: str, response: str, baseline: float, score: str = 'family', fits: int = 0, contributions: list = <factory>, shapes: list = <factory>, levels: dict = <factory>, groups: tuple = (), candidates: int = 0) -> None` | Held-out contribution and alternative-shape scores for each term. |
| `Transcript(steps: list = <factory>) -> None` | One recorded step per line of the search, in order. |
| `agent_guide(topic: str \| None = None, path_only: bool = False) -> str` | Return the agent skill text, a topic section, or a file path. |
| `api_card(obj, *, width: int = 100) -> str` | A bounded card for one function, method or class. |
| `auto_policy(data, response, *, family=None, metric='mse', complexity='balanced', folds=None, seed=0, mstop=100, df=4.0, budget=None, threshold_se=None, cache=None, progress=None, groups=(), na_action='error', unseen_levels='average', score_parameter=None, discovery_fits=100, preexploration='native', **fit_kwargs)` | Choose a specification with measured comparisons and a workflow budget. |
| `autospec(data, response, *, family=None, mstop=100, df=4.0, smooth='bbs', budget=0, folds=None, seed=0, nu=None, prior=None, monotone=None, coordinates=None, adjacency=None, basis=None, groups=None, measurements=None, treatment=None, interactions='report', deselect=True, step=None, residual_tree=False, preexploration='native', heredity='none', forbid=None, mandatory=None, joint_tree=True, exposure=None, na_action=None, lss_trees='auto', search='exhaustive')` | Guess a complete boosting specification for `data`. |
| `check_declarations(model_or_recommendation, data, *, folds=5, seed=0)` | Test each declaration of a specification against the data. |
| `classify_columns(data, response=None, df=4.0, smooth='bbs')` | Describe each predictor and name the learner suggested for it. |
| `column_screen(data, response, columns=None, *, output='auto')` | Scatterplot diagnostics of the response against each column. |
| `compare_specifications(specifications, data, response, family=None, folds=5, mstop=500, nu=0.1, seed=0, weights=None, score='family', score_parameter=None, na_action='error', observation_ids=None, include_evaluations=False, fit_budget=None, output='records', unseen_levels='average', history=None, max_mstop=None, extrapolation='warn', step='fixed')` | Score several right-hand sides on the same folds. |
| `cv_risk(rhs, data, response, family=None, folds=5, mstop=500, nu=0.1, seed=0, weights=None, score='family', na_action='error', score_parameter=None, observation_ids=None, return_result=False, retain_arrays=True, unseen_levels='average', extrapolation='warn', step='fixed', response_transform=None, max_mstop=None, mstop_grid=None, estimator='boosting', **fit_kwargs)` | Out-of-fold risk of a right-hand side. |
| `discover(data, response: str, *, newdata=None, calibration=None, family=None, seed: int = 0, level: float = 0.9, folds: int = 5, na_action: str = 'impute', budget_seconds: float \| None = None, spread: bool = True, simplify_mean: bool = True, mean_step: str = 'fixed', mean_budget: str = 'fixed', interactions: bool = True, heredity: str \| collections.abc.Mapping = 'none', forbidden_pairs=(), mandatory_terms=(), candidates: collections.abc.Mapping \| None = None, max_mstop: int \| None = None, extrapolation: str = 'warn', unseen_levels: str = 'error') -> mboostrs.discover.DiscoveryResult` | Run the rule-driven analysis of `response` on `data`. |
| `estimate_cost(call: str, *args, **kwargs) -> mboostrs.cost.CostEstimate` | Count the fits of a call and time one fit, without running the call. |
| `expand_formula(formula: str, data) -> str` | Resolve `.` and `- term` against the data, returning a plain formula. |
| `expand_interactions(formula: str, data, *, na_action: str = 'error')` | Rewrite `a %X% b` terms into buser references. |
| `expand_pairs(data, response=None, pairs=None, df=4.0, marginal_df=2.5, knots=5, kind='tensor', maxdepth=4)` | Interaction terms for `pairs`, or for every pair when none are given. |
| `holdout(data, fraction=0.2, *, groups=None, time=None, seed=0)` | Split a frame into a search part and a test part. |
| `insights(data, response, *, top_mains=8, top_pairs=5, seed=0, family=None)` | Run screen -> fit -> diagnose -> propose; return ranked findings. |
| `install_skill(dest: str \| pathlib.Path = '.claude/skills', *, overwrite: bool = False) -> pathlib.Path` | Copy the skill, its topics and examples to `dest/analyzing-data-with-mboostrs/`. |
| `label_evidence(name, values, response)` | Read a whole-number column against the response; see `LabelEvidence`. |
| `modeler_loop(frame, response='y', *, family=None, transcript=None, judgment=None, causal=None, domain_context=None, locks=(), budget=250, max_rounds=4, mstop=300, seed=0, tune_start=None, preexploration='native', workflow_budget=None, progress=None)` | Run a measured specification search with optional LLM proposals. |
| `open_search(data, response, **kwargs)` | Open a training-only session for human or external-agent tool choice. |
| `pair_term_for(left, right, *, levels=None, groups=(), categorical=(), transforms=None, data=None)` | The learner that carries an interaction between two columns. |
| `parse_formula(formula: str) -> tuple[str \| tuple[str, ...], list[str]]` | Split a linear formula into its response and predictor names. |
| `parse_gamboost_formula(formula: str, *, env: dict \| None = None, data=None) -> tuple[str \| tuple[str, str], list[mboostrs.learners.Bbs \| mboostrs.learners.Bkernel \| mboostrs.learners.Bmono \| mboostrs.learners.Bns \| mboostrs.learners.Bols \| mboostrs.learners.Brad \| mboostrs.learners.Btree \| mboostrs.learners.Bmrf \| mboostrs.learners.Bspatial \| mboostrs.learners.Brandom \| mboostrs.learners.Bconstrained \| mboostrs.learners.Buser]]` | Parse an additive formula into its response and learner objects. |
| `reference_gap(formula_or_model, data, *, family=None, folds=5, fold_scheme=None, time=None, reference='trees', maxdepth=4, max_mstop=5000, seed=0, candidates=None, top=5, lags=(1, 3, 24), nu=0.1, lag_columns=None, reference_columns='all', exclude_columns=None, extrapolation='warn', time_budget: float \| None = 600) -> mboostrs.reference_gap.ReferenceGapResult` | Compare a specification with a tree reference and rank candidate revisions. |
| `screen_discontinuities(data, target, *, response=None, columns=None, cells=20, directions=16, seed=0, pairs=None, top_k=25, all_pairs_max_p=20, fallback_top_columns=10, min_side=10, refine_units=3, refine_factor=8, type_top=5, na_action='error')` | Screen for axis-aligned and diagonal step discontinuities. |
| `screen_interactions(data, residual, response=None, bins=4, seed=0, pairs=None, groups=(), na_action='error')` | Rank candidate pairs by the residual variation they explain out of sample. |
| `select_capacity(formula, data, family=None, *, folds=5, seed=0, mstop=500, nu=0.1, pairs=3, tree=True, tree_depth=3, budget='tuned', max_mstop=None, tolerance=2.0, score='family', extrapolation='warn', estimators=('boosting',), **fit_kwargs)` | Extend a formula with interaction capacity only where it pays. |
| `sensitivity(model, data, variables=None, points=25, quantiles=(0.1, 0.9), parameter=None)` | How much the fitted function moves across each variable's range. |
| `specification_curve(source, data, response, *, budget=None, seed=0, output='auto')` | Score the specifications a search touched on one shared fold partition. |
| `suggest_specifications(report, screened=None, drop_below=0.0, top=None, model=None, noise=2.0)` | Rank candidate changes by their expected held-out gain. |
| `suggest_terms(data, response=None, df=4.0, smooth='bbs')` | The main-effect right-hand side suggested for `data`. |
| `term_report(formula, data, response=None, family=None, folds=5, mstop=500, nu=0.1, seed=0, shapes=True, df=4.0, weights=None, score='family', na_action='error', fit_budget=None, extrapolation='warn')` | Refit a specification without each term, and each term in other shapes. |

**`CapacitySelection`**

```python
pick.rungs  # one record per rung: label, formula, held-out score,
             # fold SE, whether it was selected, and why it was built
```

**`CostBudget`**

```python
budget = mb.CostBudget(fits=100, seconds=60)
```

**`DeclarationCheck`**

```python
df = mb.datasets.bodyfat()
rec = mb.autospec(df, "DEXfat", monotone={"waistcirc": "increasing"})
mb.check_declarations(rec, df, folds=3)[0].verdict
```

**`DiscoveryResult`**

```python
result.contrast("x", -1.0, 1.0).estimate
result.log[:3]  # stage, candidate, change, se, adopted, reason
```

**`Finding`**

```python
f.finding, f.effect_size, f.uncertainty, f.basis, f.evidence, f.next_action
```

**`OrderedCategoryWarning`**

```python
import warnings
warnings.simplefilter("error", mb.OrderedCategoryWarning)
```

**`ReferenceGapResult`**

```python
res.gap, res.gap_se, res.material, res.closed_share
res.to_dict()
```

**`SearchEvidence`**

```python
evidence = session.run("classify")
evidence.action, evidence.value
```

**`SearchSession`**

```python
session.state()["available_actions"]
```

**`SpecificationCurve`**

```python
curve.joint_test["p_value"], curve.label
```

**`agent_guide`**

```python
print(mb.agent_guide())
print(mb.agent_guide("index"))
print(mb.agent_guide("missing-data"))
```

**`api_card`**

```python
print(mb.api_card(mb.tune_mstop))
# or: python -m mboostrs sig tune_mstop
```

**`auto_policy`**

```python
df = mb.datasets.bodyfat()
result = mb.auto_policy(df, "DEXfat", family=mb.Gaussian(),
                        folds=3, budget=mb.CostBudget(fits=100))
```

**`autospec`**

```python
df = mb.datasets.bodyfat()
formula, learners, family, mstop = mb.autospec(df, "DEXfat")
fit = mb.gamboost(formula, data=df, learners=learners,
                   family=family, control=mb.boost_control(mstop))
```

**`check_declarations`**

```python
df = mb.datasets.bodyfat()
rec = mb.autospec(df, "DEXfat", monotone={"waistcirc": "increasing"})
mb.check_declarations(rec, df, folds=3)
```

**`column_screen`**

```python
mb.column_screen(data, "y")
```

**`compare_specifications`**

```python
table = mb.compare_specifications(
    ["bols(x1)", "bbs(x1, df=4)"],
    data=mb.datasets.heteroscedastic_lss(),
    response="y")
```

**`cv_risk`**

```python
score = mb.cv_risk("bols(x1) + bbs(x2, df=3)",
            data=mb.datasets.heteroscedastic_lss(),
            response="y", folds=5)
score
```

**`discover`**

```python
result = mb.discover(data, "y", newdata=new, seed=0)
print(result.formula, result.mean_effects)
```

**`estimate_cost`**

```python
cost = mb.estimate_cost("compare_specifications",
                        ["bols(x1)", "bbs(x1, df=4)"],
                        mb.datasets.heteroscedastic_lss(), "y")
cost.fits
```

**`holdout`**

```python
search, test = mb.holdout(data, fraction=0.2)
search_g, test_g = mb.holdout(data, fraction=0.2, groups="district")
```

**`insights`**

```python
for f in mb.insights(data, "y"):
    print(f.finding, f.basis, f.next_action)
```

**`install_skill`**

```python
mb.install_skill(".claude/skills")  # or: python -m mboostrs skill install
```

**`label_evidence`**

```python
india = mb.datasets.india(output="polars")
evidence = mb.label_evidence("mcdist", india["mcdist"],
                             india["stunting"])
evidence.labels
```

**`modeler_loop`**

```python
df = mb.datasets.bodyfat()
result = mb.modeler_loop(df, "DEXfat", family=mb.Gaussian(),
                         max_rounds=1, workflow_budget=mb.CostBudget(fits=100))
```

**`open_search`**

```python
df = mb.datasets.bodyfat()
session = mb.open_search(df, "DEXfat", family=mb.Gaussian(),
                         folds=3, score="mse", budget=mb.CostBudget(fits=60))
session.state()
```

**`pair_term_for`**

```python
mb.pair_term_for("dose", "treated",
                 levels={"dose": 200, "treated": 2})
```

**`reference_gap`**

```python
res = mb.reference_gap("y ~ bbs(x, df=4) + bols(g)", data, folds=5, seed=0)
print(res.describe())
res.candidates  # ranked terms, with measured held-out gains
```

**`select_capacity`**

```python
df = mb.datasets.bodyfat()
pick = mb.select_capacity("DEXfat ~ .", df, mstop=500)
pick.selected  # "base", "+tensors" or "+tree": the smallest rung within
                # tolerance standard errors of the best held-out score
```

**`sensitivity`**

```python
rows = mb.sensitivity(fit)
rows[0]["range of the effect"]
```

**`specification_curve`**

```python
curve = mb.specification_curve(session, df, "DEXfat", budget=200)
curve.rows  # one row per specification: gap to the selected one, with its paired SE
```

**`suggest_specifications`**

```python
report = mb.term_report(
    "y ~ bols(x1)",
    data=mb.datasets.heteroscedastic_lss(),
    folds=5)
for suggestion in mb.suggest_specifications(report):
    print(suggestion.action, suggestion.term)
```

**`term_report`**

```python
report = mb.term_report("DEXfat ~ bols(waistcirc) "
                    "+ bbs(hipcirc, df=4)",
                    data=mb.datasets.bodyfat(),
                    folds=5)
report.baseline
```


## Model selection and comparison

Compare fitted losses, information criteria, and model summaries.

| Name | What it is |
|---|---|
| `AicResult(aic: object, mstop: int, df: object, provenance: mboostrs._results.Provenance \| None = None, criterion: str = 'unspecified', normalization: str = 'unspecified', df_convention: str = 'unspecified', penalty: float \| None = None) -> None` | An information-criterion path: the value at each iteration, the iteration that minimises it, and the degrees of freedom behind it. |
| `CompareReport(spec: CompareSpec)` | Structured comparison report between Python and R. |
| `CompareSpec(model_type: str = 'glmboost', formula: str = 'y ~ x1 + x2', family: str = 'Gaussian', mstop: int = 50, nu: float = 0.1, n: int = 200, seed: int = 42, risk: str = 'inbag', weights: list[float] \| None = None)` | Specification for a single model comparison. |
| `FAMILY_SETS` |  |
| `Summary(title, header_pairs, tables=None, notes=None)` | A model summary that prints as text and displays as HTML. |
| `SummaryTable(title, headers, rows, align=None, note=None)` | A titled table of rows, rendered as text or HTML. |
| `compare_models(*models, metrics: tuple[str, ...] = ('risk', 'logLik', 'GAIC', 'mstop', 'selected_learners'), labels: list[str] \| None = None, sort: str \| None = None, cv: list \| None = None, as_frame: bool \| None = None, output: str \| None = None, common_sample: bool = False) -> list[dict[str, typing.Any]] \| typing.Any` | Compare models across multiple metrics. |
| `compare_summaries(*models, labels=None, cv=None)` | Put several fitted models side by side as one `Summary`. |
| `compare_to_r(spec: mboostrs.compare.CompareSpec, data: dict[str, numpy.ndarray] \| None = None, backend: str = 'mboost') -> mboostrs.compare.CompareReport` | Compare a mboostrs fit against R mboost. |
| `family_set(support: str) -> list[str]` | Return the candidate LSS family names for one response support. |
| `gaic_table(*models, k: float \| tuple = 2.0, labels: list[str] \| None = None, as_frame: bool \| None = None, output: str \| None = None, common_sample: bool = False) -> list[dict[str, typing.Any]] \| typing.Any` | Compare same-sample fits using an explicitly named stopping criterion. |
| `profile_mstop(model, grid: list[int] \| None = None, metric: str = 'risk', k: float = 2.0) -> list[dict[str, typing.Any]]` | Evaluate a model at different mstop values. |
| `r_squared(model, kind='both')` | Generalized R-squared for a distributional model, as `gamlss::Rsq`. |
| `scan_lss_families(formula, data, families: list \| None = None, control=None, metric: str = 'GAIC', fit_func='glmboostLSS', k: float = 2.0, support: str \| None = None, folds=None, na_action: str = 'error') -> list[dict[str, typing.Any]]` | Scan candidate LSS families and rank by a metric. |

**`FAMILY_SETS`**

```python
mb.FAMILY_SETS["realplus"]
```

**`compare_models`**

```python
report = mb.compare_models(fit_a, fit_b)
```

**`family_set`**

```python
mb.family_set("realplus")
```

**`gaic_table`**

```python
table = mb.gaic_table([fit_a, fit_b])
```

**`profile_mstop`**

```python
table = mb.profile_mstop([fit_a, fit_b])
```

**`r_squared`**

```python
mb.r_squared(fit)
```


## Effects and contrasts

Estimate prediction changes, derivatives, and contrasts over specified rows.

| Name | What it is |
|---|---|
| `ALE1DResult(variable: str, edges: numpy.ndarray, grid: numpy.ndarray, effect: numpy.ndarray, local_change: numpy.ndarray, counts: numpy.ndarray, weight_totals: numpy.ndarray, bin_index: numpy.ndarray, observation_effect: numpy.ndarray, weights: numpy.ndarray, observation_ids: numpy.ndarray, scale: str, parameter: str \| None, population: str, support: dict) -> None` | A centered univariate accumulated-local-effect curve and its support. |
| `ContrastResult(estimate: float, individual: numpy.ndarray, reference_prediction: numpy.ndarray, alternative_prediction: numpy.ndarray, weights: numpy.ndarray, observation_ids: numpy.ndarray, variable: str, reference: object, alternative: object, scale: str, parameter: str \| None, population: str, support: dict, uncertainty: str = 'none', causal: bool = False, dependence: dict \| None = None) -> None` | A standardized full-model prediction contrast and its row contributions. |
| `ElasticityResult(estimate: float, individual: numpy.ndarray, lower_value: numpy.ndarray, upper_value: numpy.ndarray, lower_prediction: numpy.ndarray, upper_prediction: numpy.ndarray, weights: numpy.ndarray, observation_ids: numpy.ndarray, variable: str, kind: str, parameter: str \| None, population: str, log_step: float, support: dict, uncertainty: str = 'none', causal: bool = False) -> None` | Average and row-specific finite log-response derivatives. |
| `MarginalEffectResult(estimate: float, individual: numpy.ndarray, lower_value: numpy.ndarray, upper_value: numpy.ndarray, lower_prediction: numpy.ndarray, upper_prediction: numpy.ndarray, weights: numpy.ndarray, observation_ids: numpy.ndarray, variable: str, scale: str, parameter: str \| None, population: str, step: float, support: dict, uncertainty: str = 'none', causal: bool = False, dependence: dict \| None = None) -> None` | Average and row-specific finite derivatives of the full prediction. |
| `PredictionProfileResult(variable: str, grid: numpy.ndarray, average: numpy.ndarray, ice: numpy.ndarray \| None, weights: numpy.ndarray, observation_ids: numpy.ndarray, scale: str, parameter: str \| None, population: str, support_fraction_far: numpy.ndarray, support: dict) -> None` | A weighted partial-dependence curve with optional ICE predictions. |
| `accumulated_local_effect(model, data, variable, *, bins=10, type='response', parameter=None, weights=None, observation_ids=None, population='supplied rows', extrapolation='error', mstop=None)` | Estimate a centered univariate accumulated local effect. |
| `average_elasticity(model, data, variable, *, kind='elasticity', log_step=0.001, parameter=None, weights=None, observation_ids=None, population='supplied rows', extrapolation='error', mstop=None)` | Average a finite elasticity or semi-elasticity of response prediction. |
| `average_marginal_effect(model, data, variable, *, step=None, type='response', parameter=None, weights=None, observation_ids=None, population='supplied rows', extrapolation='error', mstop=None, cluster=None, level=0.95, n_draws=1000, seed=0, joint_support='report', joint_threshold=1.0, support_columns=None)` | Average row-specific finite derivatives of the full fitted prediction. |
| `distributional_contrast(model, data, variable, reference, alternative, *, quantity, value=None, weights=None, observation_ids=None, population='supplied rows', extrapolation='error', mstop=None, cluster=None, level=0.95, n_draws=1000, seed=0, joint_support='report', joint_threshold=1.0, support_columns=None, refit=None)` | Contrast a fitted LSS mean, quantile, or exceedance probability. |
| `prediction_contrast(model, data, variable, reference, alternative, *, type='response', parameter=None, weights=None, observation_ids=None, population='supplied rows', extrapolation='error', mstop=None, cluster=None, level=0.95, n_draws=1000, seed=0, joint_support='report', joint_threshold=1.0, support_columns=None)` | Average a full-model prediction change over the supplied rows. |
| `prediction_profile(model, data, variable, *, grid=None, points=25, type='response', parameter=None, weights=None, observation_ids=None, population='supplied rows', extrapolation='error', retain_ice=True, support_threshold=2.5, support_columns=None, mstop=None)` | Weighted partial dependence and optional individual conditional curves. |

**`ALE1DResult`**

```python
curve = ale.to_frame(output="polars")
ale.describe()
```

**`ContrastResult`**

```python
rows = change.to_frame(output="polars")
change.describe()
```

**`ElasticityResult`**

```python
rows = elasticity.to_frame(output="polars")
elasticity.describe()
```

**`MarginalEffectResult`**

```python
rows = effect.to_frame(output="polars")
effect.describe()
```

**`PredictionProfileResult`**

```python
curve = profile.to_frame(output="polars")
profile.describe()
```

**`accumulated_local_effect`**

```python
ale = mb.accumulated_local_effect(fit, df, "waistcirc", bins=10)
```

**`average_elasticity`**

```python
elasticity = mb.average_elasticity(count_fit, df, "exposure",
                                   kind="elasticity")
```

**`average_marginal_effect`**

```python
effect = mb.average_marginal_effect(fit, df, "waistcirc",
                                    type="response")
```

**`distributional_contrast`**

```python
upper_tail = mb.distributional_contrast(
    lss_fit, df, "x", 0, 1, quantity="quantile", value=0.9,
)
```

**`prediction_contrast`**

```python
change = mb.prediction_contrast(fit, df, "waistcirc", 80, 100,
                                type="response")
```

**`prediction_profile`**

```python
profile = mb.prediction_profile(fit, df, "waistcirc", points=15)
```


## Causal estimands

Specify causal targets and adjustment assumptions, then fit treatment-effect or instrumental-variable models.

| Name | What it is |
|---|---|
| `AIPWResult(estimate: float, standard_error: float \| None, interval: tuple[float, float] \| None, level: float, spec: mboostrs.causal.CausalSpec, treatment: numpy.ndarray, outcome: numpy.ndarray, propensity: numpy.ndarray, mu0: numpy.ndarray, mu1: numpy.ndarray, scores: numpy.ndarray, influence: numpy.ndarray, fold_id: numpy.ndarray, diagnostics: dict, nuisance: dict, fold_ids: numpy.ndarray \| None = None, observed: numpy.ndarray \| None = None, response_propensity: numpy.ndarray \| None = None, weights: numpy.ndarray \| None = None, _propensity_model: object = None, _data: object = None, _groups: object = None, _ordered: bool = False) -> None` | Cross-fitted AIPW estimate with reconstructible row contributions. |
| `Assumption(statement: str, provenance: str, source: str = '', round: int \| None = None) -> None` | One causal statement with its provenance, never inferred from data. |
| `AssumptionLedger(entries: list = <factory>) -> None` | The causal statements a run depends on, tagged by who vouched for them. |
| `BLPProjection(covariates: tuple, plans: tuple, coefficients: tuple, kept: tuple, labels: tuple, n: int, level: float, recipes: dict = <factory>) -> None` | A linear projection of the effect on covariates, usable as a ranking source. |
| `CITest(x: str, y: str, given: tuple[str, ...], statistic: float, p_value: float, residual_correlation: float, n: int, folds: int, functions: int = 1, dropped: tuple[str, ...] = (), notes: tuple[str, ...] = (), skipped: bool = False, status: str = 'tested', outcome_missing: str = 'error', n_missing_outcome: int = 0) -> None` | A test of `x` independent of `y` given the columns in `given`. |
| `CausalDraft(spec: object, ledger: mboostrs.modeler.AssumptionLedger, questions: list, unresolved: list, problems: list) -> None` | A proposed causal design: a spec a person has not yet confirmed. |
| `CausalSpec(treatment: str, outcome: str, covariates: tuple[str, ...], estimand: str = 'ate', outcome_type: str = 'continuous', target_population: str = 'sample population', unit: str = 'observation', time_zero: str \| None = None, follow_up: str \| None = None, treatment_contrast: tuple[object, object] = (0, 1), post_treatment: tuple[str, ...] = (), adjustment_rationale: str = '', assumptions: tuple[str, ...] = ('consistency: each observed outcome equals the potential outcome under the observed treatment', 'conditional exchangeability: potential outcomes are independent of treatment given the baseline covariates', 'positivity: both treatment levels have positive probability across the target covariate support', "no interference: one unit's treatment does not change another unit's outcome"), propensity_clip: tuple[float, float] \| None = (0.01, 0.99), outcome_missing: str = 'error', overlap_bounds: tuple[float, float] = (0.02, 0.98)) -> None` | A binary-treatment causal question and its adjustment design. |
| `EffectModel(model: object, spec: mboostrs.causal.CausalSpec, modifiers: tuple[str, ...], terms: str, cv_rloss: dict, tau: numpy.ndarray, propensity: numpy.ndarray, outcome_mean: numpy.ndarray, diagnostics: dict, propensity_model: object = None, overlap_bounds: tuple = (0.02, 0.98), constant_rloss: float = nan, scale_plan: dict = <factory>, interval_method: str = 'refit', interval_fits: tuple = (), second_stage: object = None) -> None` | A conditional average treatment effect `tau(x)`, fitted by the R-learner. |
| `EffectModifiers(method: str, level: float, adjust: str, bounds: tuple, n: int, n_inside: int, groups: int, records: tuple, joint: tuple, _core: dict, diagnostics: dict = <factory>, message: str = '') -> None` | Per-covariate tests of effect modification, with a joint projection. |
| `IVResult(treatment: str, outcome: str, instrument: str, covariates: tuple, estimate: float, standard_error: float, level: float, interval: tuple, first_stage_r2: float, first_stage_f: float, weak_threshold: float, weak_instrument: bool, n: int, folds: int, seed: int, nuisance: dict, influence: numpy.ndarray, outcome_residual: numpy.ndarray, treatment_residual: numpy.ndarray, instrument_residual: numpy.ndarray, fold: numpy.ndarray, assumptions: tuple = ('relevance: the instrument moves the probability of treatment after the covariates', 'independence: the instrument shares no unmeasured cause with the outcome, given the covariates', 'exclusion: the instrument affects the outcome only through the treatment, given the covariates', 'monotonicity: no unit is moved out of treatment by the instrument, so the units it moves all move the same way', 'consistency: each observed outcome equals the potential outcome under the observed treatment and instrument value')) -> None` | A cross-fitted instrumental-variable estimate, its interval and its first-stage diagnostics. |
| `NegativeControlResult(kind: str, rationale: str, expected_effect: float, effect: mboostrs.causal.AIPWResult) -> None` | A prespecified negative-control estimate and its expected null value. |
| `OverlapRanking(chosen: tuple, outside: tuple, gain: float, gain_interval: tuple \| None, level: float \| None, share: float, bounds: tuple, shortfall: int, rows: tuple, tied_rows: int = 0, tie_break: str = 'row order') -> None` | The top group of rows by estimated effect, ranked inside the overlap region. |
| `OverlapReport(bounds: tuple, n: int, n_below: int, n_above: int, regions: tuple, estimand: str \| None, estimate: float \| None, standard_error: float \| None, estimate_inside: float \| None, standard_error_inside: float \| None, n_inside: int) -> None` | Rows whose estimated propensity lies outside `bounds`, where they are, and the estimate among the rows inside the bounds. |
| `TrapFinding(name: str, severity: str, message: str) -> None` | One design problem a causal claim would rest on. |
| `TrapReport(findings: tuple[mboostrs.causal.TrapFinding, ...] = (), spec: dict = <factory>) -> None` | Findings from checking a declared causal design against the supplied data. |
| `aipw(data, spec: mboostrs.causal.CausalSpec, *, folds=5, repeats=1, seed=42, smooth=False, df=4, mstop=200, max_mstop=None, tune=True, inner_folds=5, level=0.95, propensity_clip=from_spec, outcome_missing=from_spec, weights=None, outcome_model='cv', outcome_interactions=False, overlap_bounds=from_spec, target='all', propensity_terms=None, outcome_terms=None, extrapolation='warn', flat_tolerance=0.01, repeat_aggregation='average', propensity_learner='smooth', interval='influence', bootstrap_draws=200)` | Cross-fitted augmented inverse-probability estimate for binary treatment. |
| `causal_trap_report(frame, spec, *, min_per_arm=5, min_arm_share=0.01, max_missing_share=0.2, outcome_missing=from_spec)` | Check a declared causal design against the data before estimating it. |
| `ci_test(data, x, y, given=(), *, folds=5, seed=0, extrapolation='warn', outcome_missing='error')` | Test whether `x` and `y` are independent given `given`. |
| `confounding_benchmark(data, result=None, *, variables=None, folds=None, output='auto')` | Benchmark sensitivity by omitting one measured covariate at a time. |
| `confounding_sensitivity(result: mboostrs.causal.AIPWResult \| None = None, *, outcome_effects, imbalances, estimate=None, se=None, scores=None, level=0.95, output='auto')` | Bias grid for one omitted baseline confounder under an additive model. |
| `effect_model(data, spec: mboostrs.causal.CausalSpec, *, nuisance=None, modifiers=None, terms='auto', folds=5, seed=42, df=4, propensity_clip=from_spec, max_mstop=None, outcome_missing=from_spec, weights=None, overlap_bounds=from_spec, learner='r_winsorized', extrapolation='warn', flat_tolerance=0.01, propensity_learner='smooth', interval_method='auto', interval_repeats=5, bootstrap_draws=200)` | Estimate the conditional average treatment effect `tau(x)`. |
| `effect_modifiers(result, data, *, covariates=None, method='blp', groups=4, level=0.9, overlap_bounds=(0.02, 0.98), adjust='holm', gate=True, gate_seed=42)` | Test whether the effect changes along each covariate, using the AIPW scores. |
| `elicit_causal_spec(frame, judgment, *, domain_context=None)` | Interview a judgment layer for a causal design and validate it. |
| `iv_effect(data, spec: mboostrs.causal.CausalSpec, instrument, *, folds=5, seed=0, smooth=False, df=4, mstop=200, max_mstop=None, tune=True, inner_folds=5, level=0.95, weak_threshold=10.0, outcome_missing=from_spec, propensity_clip=from_spec)` | Cross-fitted partially linear instrumental-variable estimate for a binary treatment. |
| `negative_control(data, spec: mboostrs.causal.CausalSpec, *, kind, rationale, expected_effect=0.0, **aipw_options)` | Estimate a prespecified negative-control exposure or outcome effect. |
| `overlap_report(source, data=None, bounds=from_spec, *, covariates=None)` | Summarize where the estimated propensity leaves the overlap bounds. |
| `rank_by_effect(effect, newdata, *, share=0.2, propensity=None, bounds=(0.02, 0.98))` | Rank new rows by estimated effect, using only rows with overlap. |

**`AIPWResult`**

```python
rows = result.to_frame(output="polars")
result.describe()
```

**`BLPProjection`**

```python
tau_hat = proj.predict(new_rows)
ranking = mb.rank_by_effect(proj, new_rows, share=0.2, propensity=e_new)
```

**`CITest`**

```python
test.p_value, test.residual_correlation, test.describe()
```

**`CausalSpec`**

```python
design = mb.CausalSpec(
    treatment="treated", outcome="outcome",
    covariates=("age", "baseline"), estimand="ate",
    time_zero="treatment assignment", follow_up="one year",
    adjustment_rationale="age and baseline precede treatment and are common causes",
)
```

**`EffectModel`**

```python
fx.predict(new_rows)
fx.model.partial_effects()
fx.describe()
```

**`EffectModifiers`**

```python
tests.describe()
tests.projection("passing")
```

**`IVResult`**

```python
iv.describe()
iv.to_frame(output="polars")
```

**`NegativeControlResult`**

```python
check.interval_excludes_expected
check.describe()
```

**`OverlapRanking`**

```python
ranking.describe()
ranking.table(output="polars")
```

**`OverlapReport`**

```python
report.describe()
report.regions
```

**`aipw`**

```python
result = mb.aipw(data, design, folds=5, mstop=200)
```

**`ci_test`**

```python
mb.ci_test(data, "proxy", "y", given=["driver"]).describe()
```

**`confounding_benchmark`**

```python
benchmarks = mb.confounding_benchmark(data, result,
                                      variables=["age", "baseline"])
```

**`confounding_sensitivity`**

```python
grid = mb.confounding_sensitivity(result, outcome_effects=[0, 1],
                                   imbalances=[0, 0.5])
```

**`effect_model`**

```python
fx = mb.effect_model(data, design, nuisance=result)
```

**`effect_modifiers`**

```python
tests = mb.effect_modifiers(result, data, method="blp", level=0.9)
tests.table(output="polars"), tests.selected()
```

**`iv_effect`**

```python
iv = mb.iv_effect(data, design, "offer", folds=5, mstop=200)
iv.estimate, iv.interval, iv.first_stage_f
```

**`negative_control`**

```python
check = mb.negative_control(data, control_design, kind="outcome",
                            rationale="measured before treatment",
                            folds=5, mstop=200)
```

**`overlap_report`**

```python
report = mb.overlap_report(result, data=data, bounds=(0.02, 0.98))
report.share_outside, report.estimate, report.estimate_inside
```

**`rank_by_effect`**

```python
ranking = mb.rank_by_effect(fx, new_rows, share=0.2)
ranking.chosen, ranking.outside, ranking.gain_interval
```


## Persistence and reporting

Save fitted models, inspect recorded decisions, and report predictions.

| Name | What it is |
|---|---|
| `DecisionTrail(schema_version: str = '1.0', provenance: dict[str, typing.Any] = <factory>, data: dict[str, typing.Any] = <factory>, folds: dict[str, typing.Any] = <factory>, criterion: str = 'oof_log_score', steps: list[dict[str, typing.Any]] = <factory>, final: dict[str, typing.Any] = <factory>, uncertainty: dict[str, typing.Any] = <factory>, budget: dict[str, typing.Any] = <factory>) -> None` | A builder for the record in section 5 of the design note. |
| `ExecutionReport(schema_version: int, instrumentation: str, route: str, n_observations: int, n_learners: int, total_basis_columns: int, requested_mstop: int, completed_mstop: int, effective_threads: int, gram_eligible: bool, fallback_reasons: tuple[str, ...] = (), scalar_learner_count: int = 0, block_learner_count: int = 0, tree_learner_count: int = 0, cholesky_solver_count: int = 0, lu_solver_count: int = 0, regularized_solver_count: int = 0, max_solver_jitter: float \| None = None, factor_cache_bytes: int = 0, retained_learner_bytes: int = 0, gram_cache_bytes: int = 0, calibration_count: int = 0, tie_selection_count: int = 0, component_subsample_iterations: int = 0, calibration_converged_count: int = 0, calibration_limit_count: int = 0, calibration_failure_count: int = 0, nonfinite_candidate_fallbacks: int = 0, setup_ns: int \| None = None, basis_ns: int \| None = None, factorization_ns: int \| None = None, cross_gram_ns: int \| None = None, iteration_ns: int \| None = None, reconstruction_ns: int \| None = None, binding_ns: int \| None = None, total_ns: int \| None = None) -> None` | Versioned summary of the numerical route used by a fitted model. |
| `PathRow(iteration: int, cycle: int \| None, parameter_index: int \| None, parameter: str \| None, learner_index: int, learner: str, risk: float \| None, risk_change: float \| None, coefficient_norm: float, scaled_coefficient_norm: float, nuisance: tuple[float, ...] = ()) -> None` | One learner update in a fitted boosting path. |
| `ReproductionResult(fit: mboostrs._procedure._fit.Fit, report: str, matched: bool, differences: tuple[str, ...], prediction_sha256: str, original_trail: dict[str, typing.Any], reproduced_trail: dict[str, typing.Any]) -> None` | Outcome of replaying a saved `DecisionTrail` on a dataset. |
| `build_info() -> collections.abc.Mapping[str, typing.Any]` | Return immutable compile-time identity for the loaded Rust extension. |
| `dump_result_json(data: Any, *, indent: int \| None = None) -> str` | Serialize a result dictionary or JSON data with reversible nonfinite values. |
| `llm_describe(model, *, llm='nvidia/nemotron-3-ultra-550b-a55b', base_url=None, api_key=None, client=None, reasoning=False, num_sentences=7, max_tokens=4000, temperature=0.7, timeout=180, return_conversation=False, **prompt_kwargs)` | Send the model's description to a language model and return the reply. |
| `llm_explain_prompt(model, *, which=None, top=None, n_points=12, cells=5, parameter=None, dataset_description=None, response_description=None, task=None, cv=None, stabsel=None, include=('reading', 'facts', 'importance', 'effects', 'caveats', 'task'), expert='an expert statistician and applied data analyst', output='text', num_sentences=7)` | Render a fitted model as a prompt for a language model. |
| `load_model(path)` | Read a model artifact written by `save_model`. |
| `load_result_json(document: str) -> Any` | Decode a `dump_result_json` document into plain Python result data. |
| `provenance(data: Any = None, *, seed: int \| None = None) -> dict[str, typing.Any]` | Return build, environment and optional data identity metadata for a result. |
| `report(fit_obj: Any, format: str = 'text') -> Any` | Render a `Fit` (or `DecisionTrail`) as text, markdown or JSON. |
| `reproduce(trail: mboostrs._procedure._fit.Fit \| mboostrs._procedure._trail.DecisionTrail \| dict[str, typing.Any] \| str \| pathlib.Path, data: Any, *, report_format: str = 'markdown', verify: bool = True, **overrides: Any) -> mboostrs._procedure._reproduce.ReproductionResult` | Reproduce a recorded `fit` and its discovery report in one call. |
| `save_model(model, path, *, scoring_only=False, indent: int \| None = None) -> pathlib.Path` | Write `model` to `path` as a schema-versioned JSON artifact. |

**`DecisionTrail`**

```python
m = mb.fit(mb.datasets.bodyfat(), "DEXfat", seed=0)
restored = mb.DecisionTrail.from_dict(m.trail.to_json())
restored.to_dict() == m.trail.to_dict()
```

**`ReproductionResult`**

```python
check.matched, check.differences, check.report[:120]
```

**`build_info`**

```python
identity = dict(mb.build_info())
identity["build_id"]
```

**`dump_result_json`**

```python
text = mb.dump_result_json({"width": float("inf"), "missing": None})
```

**`llm_describe`**

```python
reply = mb.llm_describe(fit)
```

**`llm_explain_prompt`**

```python
print(fit.llm_explain_prompt(include=("importance",)))
```

**`load_model`**

```python
fit = mb.load_model("fit.json")
fit.unavailable_operations()
```

**`load_result_json`**

```python
text = mb.dump_result_json({"width": float("inf")})
record = mb.load_result_json(text)
```

**`provenance`**

```python
record = mb.provenance({"y": [1.0, 2.0, 3.0]}, seed=1)
record["data"]["sha256"]
```

**`report`**

```python
m = mb.fit(mb.datasets.bodyfat(), "DEXfat", seed=0)
print(mb.report(m))              # text
mb.report(m, format="json")["final"]  # the trail dict itself
```

**`reproduce`**

```python
fitted = mb.fit(df, "y", seed=0)
check = mb.reproduce(fitted.trail, df)
check.matched, check.prediction_sha256
```

**`save_model`**

```python
mb.save_model(fit, "fit.json")
```


## Capabilities

List supported operations and the tests used to validate them.

| Name | What it is |
|---|---|
| `capabilities(*, as_frame: bool = True, output: str \| None = None) -> dict[str, typing.Any]` | Return family, learner, and Python-method capability tables. |
| `family_capabilities(*, as_frame: bool = True, output: str \| None = None) -> Any` | Return family support and evidence rows for both frontends. |
| `learner_capabilities(*, as_frame: bool = True, output: str \| None = None) -> Any` | Return base-learner support and evidence rows for both frontends. |
| `method_capabilities(*, frontend: str = 'python', as_frame: bool = True, output: str \| None = None) -> Any` | Return public method support for `frontend='python'` or `'r'`. |
| `validation_capabilities(*, as_frame: bool = True, output: str \| None = None) -> Any` | Return the explicit validation ledger behind advertised capabilities. |

**`capabilities`**

```python
caps = mb.capabilities(output="polars")
```

**`family_capabilities`**

```python
caps = mb.family_capabilities(output="polars")
```

**`learner_capabilities`**

```python
caps = mb.learner_capabilities(output="polars")
```

**`method_capabilities`**

```python
caps = mb.method_capabilities(output="polars")
```

**`validation_capabilities`**

```python
vals = mb.validation_capabilities(output="polars")
```


## Errors

What a failed call raises. Each class subclasses the built-in exception it replaces and carries `code`, `subject`, `details` and `rust_variant` attributes.

| Name | What it is |
|---|---|
| `MboostrsError(message: str = '', *, code: str \| None = None, subject: str \| None = None, details: str \| None = None, rust_variant: str \| None = None)` | Base class for every mboostrs error, carrying error attributes. |
| `MboostrsKeyError(message: str = '', *, code: str \| None = None, subject: str \| None = None, details: str \| None = None, rust_variant: str \| None = None)` | A missing map key, such as a column the model needs but the data lacks. |
| `MboostrsRuntimeError(message: str = '', *, code: str \| None = None, subject: str \| None = None, details: str \| None = None, rust_variant: str \| None = None)` | A numerical or solver failure in the Rust engine. |
| `MboostrsValueError(message: str = '', *, code: str \| None = None, subject: str \| None = None, details: str \| None = None, rust_variant: str \| None = None)` | A failed validation: a bad argument, shape, parameter, or response. |

**`MboostrsError`**

```python
try:
    fit = mb.gamboost("y ~ bbs(x)", data=data_faulty)
except mb.MboostrsError as error:
    print(error.code, error.details)
```

**`MboostrsKeyError`**

```python
try:
    fit = mb.glmboost("y ~ zzz", data=data_faulty)
except mb.MboostrsKeyError as error:
    print(error.code, error.subject)
```

**`MboostrsRuntimeError`**

```python
try:
    fit = mb.gamboost("y ~ bbs(x)", data=data_faulty)
except mb.MboostrsRuntimeError as error:
    print(error.rust_variant)
```

**`MboostrsValueError`**

```python
try:
    fit = mb.glmboost("y ~ x", data=data_faulty,
                       control=mb.boost_control(10, nu=3))
except mb.MboostrsValueError as error:
    print(error.code, error.subject)
```


## Utilities

| Name | What it is |
|---|---|
| `collect_warnings()` | Show each repeated warning once, with a count, when the block ends. |
| `datasets` | Built-in datasets for mboostrs examples and case studies. |
| `plot` | Plotting functions for mboostrs models. |
| `weighted_median(x, w=1.0, na_rm=False)` | The weighted median, matching `gamboostLSS:::weighted.median`. |

**`collect_warnings`**

```python
with mb.collect_warnings():
    for seed in range(20):
        mb.ci_test(data, "x", "y", given=["z"], seed=seed)
```

**`weighted_median`**

```python
mb.weighted_median(values, weights)
```


(the-model-surface)=

## Model methods and properties

The table identifies each public member as a property or a method for each fitted-model class. Access a property without parentheses; call a method with parentheses. `--` marks an unavailable member. Check the model class before writing code that handles several kinds of fit.

| Member | glmboost | gamboost | LSS |
|---|---|---|---|
| `PI` | -- | -- | method — Alias for `prediction_interval`, matching R `PI` naming. |
| `aic` | method — Compute AIC for each boosting iteration. | method — Compute AIC for each boosting iteration. | -- |
| `base_learner_names` | method — Names of all base-learners (including intercept if present). | method — Names of all base learners. | method — Base-learner names, as a `{parameter: [name, ...]}` mapping. |
| `binning_report` | -- | method — What binning each learner actually got, and what it cost. | -- |
| `bnames` | method — Alias for `base_learner_names`. | method — Alias for `base_learner_names`. | method — Alias for `base_learner_names`. |
| `btree_partition_table` | -- | method — The fitted btree partition as a Polars frame, one row per region. | -- |
| `btree_splits` | -- | method — The cutpoints the fitted btree learners actually used. | -- |
| `calibration_frame` | method — Calibration table. See `mboostrs.calibration_frame`. | method — Calibration table. See `mboostrs.calibration_frame`. | method — Calibration table. See `mboostrs.calibration_frame`. |
| `centile_fan_data` | -- | -- | method — Observed responses and fitted centiles ordered by `xvar`. |
| `centiles` | -- | -- | method — Distribution centile table for fitted LSS parameters. |
| `check_binning` | -- | method — Refit one subsample with binning on and off. | -- |
| `coef` | method — Extract coefficients per predictor. | method — Extract coefficients per base-learner. | method — Per-parameter aggregated coefficients (sum of nu*coef per learner). |
| `coef_all` | method — Coefficients for every learner. See `mboostrs.coef_all`. | method — Coefficients for every learner. See `mboostrs.coef_all`. | method — All-parameter coefficient table, analogous to GAMLSS `coefAll`. |
| `coef_path` | method — Compute cumulative coefficient paths over boosting iterations. | -- | -- |
| `coefficients_` | property | -- | -- |
| `concurvity` | method — Concurvity of the fitted terms, as mgcv's `concurvity` measures it. | method — Concurvity of the fitted terms, as mgcv's `concurvity` measures it. | -- |
| `confint` | method — Bootstrap confidence intervals for coefficients. | method — Confidence intervals for partial effects. | -- |
| `conformal_interval` | method — Distribution-free prediction interval. See `mboostrs.conformal_interval`. | method — Distribution-free prediction interval. See `mboostrs.conformal_interval`. | -- |
| `continue_to` | method — Return the model grown to a larger `mstop`. | method — Return the model grown to a larger `mstop`. | method — Return the model grown to a larger `mstop`, per parameter. |
| `control` | property — The `boost_control` this model was fitted with. | property — The `boost_control` this model was fitted with. | property — The `boost_control` this model was fitted with. |
| `cvrisk` | method — Cross-validated risk along the path. | method — Cross-validate the fitted model's risk path. | method — Cross-validated risk for this model's own specification. |
| `density_overlay_data` | -- | -- | method — Observed and fitted marginal-distribution curves as a table. |
| `deviance_increment` | method — Per-observation deviance. See `mboostrs.deviance_increment`. | method — Per-observation deviance. See `mboostrs.deviance_increment`. | method — Per-observation deviance contribution for supported LSS families. |
| `deviance_profile` | -- | -- | method — Binned joint deviance contributions along a numeric covariate. |
| `execution_report` | method — Return the versioned numerical execution report for this fit. | method — Return the versioned numerical execution report for this fit. | method — Return aggregate numerical-route metadata for this LSS fit. |
| `extract` | method — Extract model components, matching R's extract.glmboost. | method — Extract model components, matching R's extract.mboost/glmboost. | -- |
| `extrapolation_report` | method — Predictors with rows of `newdata` outside their training range. | method — Terms and covariates with rows outside their training range. | method — Terms and covariates with rows outside their training range, by parameter. |
| `family` | property — The response family the model was fitted with. | property — The response family the model was fitted with. | -- |
| `fit_warnings` | property — Warnings the fit showed, as `warnings.WarningMessage` objects. | property — Warnings the fit showed, as `warnings.WarningMessage` objects. | property — Warnings the fit showed, as `warnings.WarningMessage` objects. |
| `fitted` | method — Fitted values, optionally at a given mstop. | method — Fitted values, optionally at a given mstop. | method — In-sample fitted values, mirroring `predict(newdata=None, ...)`. |
| `fitted_` | property — The fitted linear predictor. | property — The fitted linear predictor. | property — Fitted link-scale values per parameter, as a mapping. |
| `formula` | property — The formula as it was written. | property — The formula as it was written (`y ~ .` stays `y ~ .`). `variable_names` lists the columns it expanded to. | -- |
| `gaic` | -- | -- | method — Update-count penalized likelihood heuristic for LSS models. |
| `hatvalues` | method — Per-observation leverage: the hat-matrix diagonal at the current mstop, as a 1-d array of length n. | method — Per-observation leverage: the hat-matrix diagonal at the current mstop, as a 1-d array of length n. | -- |
| `input_weights` | method — The weights as the caller passed them, or `None` if unweighted. | method — The weights as the caller passed them, or `None` if unweighted. | method — The weights as the caller passed them, or `None` if unweighted. |
| `learner_index` | -- | method — Index of a base learner, disambiguating learners that share a name. | -- |
| `likelihood_contract` | method — What this family's risk is: a likelihood, a loss, or neither. | method — What this family's risk is: a likelihood, a loss, or neither. | method — What this family's risk is: a likelihood, a loss, or neither. |
| `llm_describe` | method — Send this fit's description to a language model and return the reply. | method — Send this fit's description to a language model and return the reply. | method — Send this fit's description to a language model and return the reply. |
| `llm_explain_prompt` | method — Render this fit as a prompt for a language model. | method — Render this fit as a prompt for a language model. | method — Render this fit as a prompt for a language model. |
| `logLik` | method — The fitted model's log-likelihood, or a refusal naming the reason. | method — The fitted model's log-likelihood, or a refusal naming the reason. | method — Joint log-likelihood at the current LSS fit. |
| `mean` | -- | -- | method — Expected response under the fitted distribution. |
| `model_frame` | method — Return the data used for fitting as a dict of arrays. | method — Return the data used for fitting as a dict of arrays. | -- |
| `model_matrix` | method — Return the design matrix (n × p) for the specified predictors. | method — Return the design matrix for the specified base-learner(s). | -- |
| `model_weights` | method — Return observation weights used for fitting. | method — Return observation weights used for fitting. | method — Observation weights used for fitting. |
| `mstop` | property | property | property |
| `negative_risk` | method — The negated empirical risk at the current mstop. | method — The negated empirical risk at the current mstop. | method — The negated joint risk at the current fit. |
| `nuisance` | method — The family's nuisance parameter at the model's step count, as R's `nuisance()` returns it (the size of `NBinomial`, `Var = mu + mu**2 / sigma`, the shape of `GammaReg`, the thresholds of `PropOdds`); None when there is none. | method — Return nuisance parameters (family-specific). | -- |
| `offset` | property — The model offset: the constant the boosting path started from. | property — The model offset: the constant the boosting path started from. | property — Per-parameter offsets, as `{parameter: value}`. |
| `offset_` | property — The scalar model offset; the same value as `offset`. | property — The scalar model offset; the same value as `offset`. | property — Per-parameter offsets, as a mapping. See also `offset`. |
| `outlier_frame` | method — Flagged outliers. See `mboostrs.outlier_frame`. | method — Flagged outliers. See `mboostrs.outlier_frame`. | -- |
| `parameter_risk` | -- | -- | method — The joint risk after each parameter's own updates, as gamboostLSS's `risk(merge = FALSE)` and `risk(merge = TRUE)`. |
| `params` | -- | -- | method — Predicted LSS parameters for all family parameters. |
| `parity_notes` | method — Where this fit differs from what its names refer to. | method — Where this fit differs from what its names refer to. | method — Where this fit differs from what its names refer to. |
| `partial_effects` | -- | method — Compute per-learner partial effects (contribution to the additive predictor). | method — Return formula-ordered learner contributions for one LSS parameter. |
| `path_frame` | method — Return one structured row per boosting iteration. | method — Return one structured row per boosting iteration. | method — Return globally ordered parameter-update rows for this LSS fit. |
| `penalty_drift` | -- | method — Each penalized learner's degrees of freedom along the fitted path. | -- |
| `pit` | method — Probability integral transform. See `mboostrs.pit`. | method — Probability integral transform. See `mboostrs.pit`. | method — Probability integral transform values for fitted LSS distributions. |
| `plot` | method — Plot coefficient paths. | method — Plot partial effects. | -- |
| `plot_btree_partition` | -- | method — Plot a btree term's fitted partition. | -- |
| `plot_centile_fan` | -- | -- | method — Plot nested fitted centile bands against `xvar`. |
| `plot_coef_path` | method — Plot coefficient paths. | -- | -- |
| `plot_density` | -- | -- | method — Overlay the observed and average fitted response distributions. |
| `plot_deviance_profile` | -- | -- | method — Plot binned joint deviance along a covariate or row order. |
| `plot_partial_effects` | -- | method — Plot partial effects. | -- |
| `plot_residuals` | method — Plot residuals vs fitted values. | method — Plot residuals vs fitted values. | -- |
| `plot_risk` | method — Plot the risk path. | method — Plot the risk path. | method — Plot the joint risk path. |
| `plot_selected` | method — Plot selection frequency per base-learner. | method — Plot selection frequency per base-learner. | method — Plot selection frequency per base learner and parameter. |
| `plot_varimp` | method — Plot variable importance as a horizontal bar chart. | method — Plot variable importance as a horizontal bar chart. | -- |
| `predict` | method — Predicted values, optionally decomposed by learner. | method — Predicted values, optionally decomposed by learner. | method — Predictions for one distribution parameter, or for all of them. |
| `predict_all` | method — Per-learner predictions. See `mboostrs.predict_all`. | method — Per-learner predictions. See `mboostrs.predict_all`. | method — All-parameter prediction table, analogous to GAMLSS `predictAll`. |
| `predict_over_grid` | -- | method — Predictions at each value of one variable, other columns held per row. | -- |
| `predict_over_mstop` | -- | method — Predictions at several path positions, in one pass over the basis. | -- |
| `predict_parameters` | method — Return the single fitted predictor as `{"f": array}`. | method — Return the single fitted predictor as `{"f": array}`. | method — Return response-scale predictions by distribution parameter. |
| `predict_path` | method — Predicted values at every `mstop` in `grid`, in one pass. | -- | -- |
| `prediction_contributions` | method — Per-learner prediction contributions. | method — Per-learner prediction contributions. | -- |
| `prediction_interval` | -- | -- | method — Central prediction interval table for the fitted LSS distribution. |
| `predint` | -- | -- | method — Alias for `prediction_interval`, matching R `predint` naming. |
| `profile_mstop` | method — Metric over the mstop path. See `mboostrs.profile_mstop`. | method — Metric over the mstop path. See `mboostrs.profile_mstop`. | method — Refit over a grid of stopping points and report `metric` at each. |
| `q_stats` | method — Binned quantile-residual statistics. See `mboostrs.q_stats`. | method — Binned quantile-residual statistics. See `mboostrs.q_stats`. | method — Binned quantile-residual moment summaries. |
| `quantile_residuals` | method — Randomized quantile residuals. See `mboostrs.quantile_residuals`. | method — Randomized quantile residuals. See `mboostrs.quantile_residuals`. | method — Randomized quantile residuals for fitted LSS distributions. |
| `range_report` | -- | method — Each learner's training range against `newdata`. | -- |
| `residual_frame` | method — Residuals as a frame. See `mboostrs.residual_frame`. | method — Residuals as a frame. See `mboostrs.residual_frame`. | method — Diagnostic residual table for LSS models. |
| `residual_kinds` | method — The residual kinds defined for this model's family. | method — The residual kinds defined for this model's family. | method — Residual kinds defined for the fitted LSS family. |
| `residuals` | method — Residuals of the named kind, or a refusal saying why there are none. | method — Residuals of the named kind, or a refusal saying why there are none. | method — Residuals of the named kind for an LSS parameter, or a refusal. |
| `risk` | property — The empirical risk after each iteration, as a read-only array. | property — The empirical risk after each iteration, as a read-only array. | property — The joint risk after each iteration (each full cycle for a cyclic fit, each update for a non-cyclic one), as a read-only array. |
| `risk_` | property — The empirical risk after each iteration, length `mstop + 1`. | property — The empirical risk after each iteration, length `mstop + 1`. | property — The joint risk after each iteration. |
| `save` | method — Write a portable JSON artifact. See `mboostrs.save_model`. | method — Write a portable JSON artifact. See `mboostrs.save_model`. | method — Write a portable JSON artifact. See `mboostrs.save_model`. |
| `selected` | property — Learner names selected at each iteration. | property — Learner names selected at each iteration. | property — Selected base-learners at each iteration, as `{parameter: path}`. |
| `selected_terms` | method — One row per boosting update: `order` (1-based, across the whole fit), `parameter` (`"f"` here), `iteration` (1-based, within the parameter; equal to `order` for a standard model) and `learner` (the name `base_learner_names` gives it). | method — One row per boosting update: `order` (1-based, across the whole fit), `parameter` (`"f"` here), `iteration` (1-based, within the parameter; equal to `order` for a standard model) and `learner` (the name `base_learner_names` gives it). | method — One row per boosting update, in the order the fit made them: `order` (1-based, across all parameters), `parameter`, `iteration` (1-based, within the parameter) and `learner`. A cyclic fit updates every parameter once per iteration, in parameter order; a non-cyclic fit records the parameter it chose at each update. |
| `selected_which` | method — Learner names selected at each iteration, optionally filtered. | method — Learner names selected at each iteration, optionally filtered. | method — Selected learner names per parameter, optionally filtered. |
| `selection_counts` | -- | method — One row per base learner with the boosting updates it received. | -- |
| `size` | property — Fitted NBinomial size, with `Var(Y) = mu + mu**2 / size`. | property — Fitted NBinomial size, with `Var(Y) = mu + mu**2 / size`. | -- |
| `stabsel` | method — Stability selection. See `mboostrs.stabsel` for the arguments (`cutoff`, `q`, `PFER`, `B`, `folds`, ...). | method — Stability selection. See `mboostrs.stabsel` for the arguments (`cutoff`, `q`, `PFER`, `B`, `folds`, ...). | method — Stability selection for this model's own specification. |
| `stopping_budget` | method — How far this model was boosted; see `StoppingBudget`. | method — How far this model was boosted; see `StoppingBudget`. | method — How far this model was boosted, as `mboostrs._model_api.StoppingBudget`: per-parameter update counts, their total (the budget a non-cyclic refit takes) and whether the parameters were updated cyclically or non-cyclically. |
| `stopping_report` | method — Where the boosting path stopped, and whether that looks finished. | method — Where the boosting path stopped, and whether that looks finished. | method — Where the boosting path stopped, and whether that looks finished. |
| `summary` | method — Summarise the fitted model. | method — Summarise the fitted model. | method — Summarise the fitted LSS model. |
| `support_report` | method — Whether each new row lies where the training data has rows. | method — Whether each new row lies where the training data has rows. | -- |
| `survfit` | method — Breslow baseline-survival curve for a fitted CoxPH model. | method — Breslow baseline-survival curve for a fitted CoxPH model. | -- |
| `survival_at` | method — Predicted survival probability S(t \| x) at fixed times, per subject. | method — Predicted survival probability S(t \| x) at fixed times, per subject. | -- |
| `to_dict` | method — The `summary(as_dict=True)` record plus a `"provenance"` entry: the library build and environment, and the fingerprint of the training data. `include_arrays` is accepted for the uniform export signature; the summary holds no per-row arrays. | method — The `summary(as_dict=True)` record plus a `"provenance"` entry: the library build and environment, and the fingerprint of the training data. `include_arrays` is accepted for the uniform export signature; the summary holds no per-row arrays. | method — The `summary(as_dict=True)` record plus a `"provenance"` entry: the library build and environment, and the fingerprint of the training data. `include_arrays` is accepted for the uniform export signature; the summary holds no per-row arrays. A model loaded from an artifact keeps no training frame, so its provenance has no `"data"` entry. |
| `unavailable_operations` | method — Operations this model cannot perform, and why. | method — Operations this model cannot perform, and why. | -- |
| `update` | method — Update the model by changing mstop or other parameters. | method — Update the model by changing mstop or other parameters. | method — Refit the stored LSS model specification. |
| `variable_names` | method — Names of all variables used in the model (response + predictors). | method — Names of all variables used in the model (response + predictors). | method — Distinct predictor names, in formula order. |
| `varimp` | method — Variable importance based on in-bag risk reduction per predictor. | method — Variable importance based on in-bag risk reduction per base-learner. | method — Variable importance from in-bag risk reduction, per parameter. |
| `weight_policy` | property — The named policy that settled the weight scale for this fit. | property — The named policy that settled the weight scale for this fit. | property — The named policy that settled the weight scale for this fit. |
| `weights` | property — The effective weight vector the fit ran on. | property — The effective weight vector the fit ran on. | property — The effective weight vector the fit ran on. |
| `weights_note` | method — One line saying how the weight scale was settled. | method — One line saying how the weight scale was settled. | method — One line saying how the weight scale was settled. |
| `with_mstop` | method — Return a copy truncated to `k` boosting iterations. | method — Return a copy truncated to `k` boosting iterations. | method — Return a copy refit to `k` boosting iterations. |
| `worm_data` | method — Worm-plot data. See `mboostrs.worm_data`. | method — Worm-plot data. See `mboostrs.worm_data`. | method — Worm-plot data. See `mboostrs.worm_data`. |
| `worm_plot` | -- | -- | method — Worm plot(s) of (randomized) quantile residuals (R's `gamlss::wp`). |

(the-rust-extension-surface)=

## Compiled extension functions

The `mboostrs._rust` module is the compiled engine boundary, rendered here from `python/mboostrs/_rust.pyi` (itself generated from the compiled module's signatures). **These are not the public API**. Production GAM fitting and resampling use compact learner-spec calls; the wide `fit_gamboost` and `cvrisk_gamboost` entries are deprecated compatibility symbols scheduled for removal in 0.2. They remain visible so a reader debugging across versions can identify the boundary in use. Argument types are `Any` because PyO3 does not surface them. For the crate's Rust-side public items, see the crate README and `cargo doc -p mboostrs-core`; for R, `help(package = "mboostrs")`.

| Symbol | Signature |
|---|---|
| `GamRefitBootstrapCore` | `GamRefitBootstrapCore` |
| `GamRefitBootstrapCore.converged` | `GamRefitBootstrapCore.converged` — `converged(self) -> Any:` |
| `GamRefitBootstrapCore.linear_predictor` | `GamRefitBootstrapCore.linear_predictor` — `linear_predictor(self, raws: Any, bys: Any, zero_rows: Any) -> Any:` |
| `GamRefitBootstrapCore.n_draws` | `GamRefitBootstrapCore.n_draws` — `n_draws(self, *args: Any, **kwargs: Any) -> Any:` |
| `GamRefitCore` | `GamRefitCore` |
| `GamRefitCore.blocks` | `GamRefitCore.blocks` — `blocks(self) -> Any:` |
| `GamRefitCore.coef` | `GamRefitCore.coef` — `coef(self, *args: Any, **kwargs: Any) -> Any:` |
| `GamRefitCore.contrast` | `GamRefitCore.contrast` — `contrast(self, hi_raws: Any, hi_bys: Any, hi_zero: Any, lo_raws: Any, lo_bys: Any, lo_zero: Any, corrected: Any = False) -> Any:` |
| `GamRefitCore.convergence` | `GamRefitCore.convergence` — `convergence(self) -> Any:` |
| `GamRefitCore.cov` | `GamRefitCore.cov` — `cov(self, *args: Any, **kwargs: Any) -> Any:` |
| `GamRefitCore.cov_corrected` | `GamRefitCore.cov_corrected` — `cov_corrected(self, *args: Any, **kwargs: Any) -> Any:` |
| `GamRefitCore.df` | `GamRefitCore.df` — `df(self, *args: Any, **kwargs: Any) -> Any:` |
| `GamRefitCore.edf_total` | `GamRefitCore.edf_total` — `edf_total(self, *args: Any, **kwargs: Any) -> Any:` |
| `GamRefitCore.effect` | `GamRefitCore.effect` — `effect(self, b: Any, raw: Any, by: Any = None, zero_rows_have_no_effect: Any = False, corrected: Any = False) -> Any:` |
| `GamRefitCore.intercept` | `GamRefitCore.intercept` — `intercept(self, *args: Any, **kwargs: Any) -> Any:` |
| `GamRefitCore.link` | `GamRefitCore.link` — `link(self, *args: Any, **kwargs: Any) -> Any:` |
| `GamRefitCore.loglik` | `GamRefitCore.loglik` — `loglik(self, *args: Any, **kwargs: Any) -> Any:` |
| `GamRefitCore.n` | `GamRefitCore.n` — `n(self, *args: Any, **kwargs: Any) -> Any:` |
| `GamRefitCore.prediction` | `GamRefitCore.prediction` — `prediction(self, raws: Any, bys: Any, zero_rows: Any, corrected: Any = False) -> Any:` |
| `GamRefitCore.reml_score` | `GamRefitCore.reml_score` — `reml_score(self, *args: Any, **kwargs: Any) -> Any:` |
| `GamRefitCore.residual_df` | `GamRefitCore.residual_df` — `residual_df(self, *args: Any, **kwargs: Any) -> Any:` |
| `GamRefitCore.rho` | `GamRefitCore.rho` — `rho(self, *args: Any, **kwargs: Any) -> Any:` |
| `GamRefitCore.rho_hessian` | `GamRefitCore.rho_hessian` — `rho_hessian(self, *args: Any, **kwargs: Any) -> Any:` |
| `GamRefitCore.scale` | `GamRefitCore.scale` — `scale(self, *args: Any, **kwargs: Any) -> Any:` |
| `GamRefitCore.theta` | `GamRefitCore.theta` — `theta(self, *args: Any, **kwargs: Any) -> Any:` |
| `GamRefitCore.to_json` | `GamRefitCore.to_json` — `to_json(self) -> Any:` |
| `GamlssRefitCore` | `GamlssRefitCore` |
| `GamlssRefitCore.coef` | `GamlssRefitCore.coef` — `coef(self, *args: Any, **kwargs: Any) -> Any:` |
| `GamlssRefitCore.convergence` | `GamlssRefitCore.convergence` — `convergence(self) -> Any:` |
| `GamlssRefitCore.cov` | `GamlssRefitCore.cov` — `cov(self, *args: Any, **kwargs: Any) -> Any:` |
| `GamlssRefitCore.cov_corrected` | `GamlssRefitCore.cov_corrected` — `cov_corrected(self, *args: Any, **kwargs: Any) -> Any:` |
| `GamlssRefitCore.edf_total` | `GamlssRefitCore.edf_total` — `edf_total(self, *args: Any, **kwargs: Any) -> Any:` |
| `GamlssRefitCore.effect` | `GamlssRefitCore.effect` — `effect(self, q: Any, b: Any, raw: Any, by: Any = None, zero_rows_have_no_effect: Any = False, corrected: Any = False) -> Any:` |
| `GamlssRefitCore.intercept` | `GamlssRefitCore.intercept` — `intercept(self, q: Any) -> Any:` |
| `GamlssRefitCore.parameter_contrasts` | `GamlssRefitCore.parameter_contrasts` — `parameter_contrasts(self, hi_raws: Any, hi_bys: Any, hi_zero: Any, lo_raws: Any, lo_bys: Any, lo_zero: Any, corrected: Any = False) -> Any:` |
| `GamlssRefitCore.parameter_draws` | `GamlssRefitCore.parameter_draws` — `parameter_draws(self, raws: Any, bys: Any, zero_rows: Any, n_draws: Any = 1000, seed: Any = 0, corrected: Any = False) -> Any:` |
| `GamlssRefitCore.parameter_predictions` | `GamlssRefitCore.parameter_predictions` — `parameter_predictions(self, raws: Any, bys: Any, zero_rows: Any, corrected: Any = False) -> Any:` |
| `GamlssRefitCore.params` | `GamlssRefitCore.params` — `params(self) -> Any:` |
| `GamlssRefitCore.reml_score` | `GamlssRefitCore.reml_score` — `reml_score(self, *args: Any, **kwargs: Any) -> Any:` |
| `GamlssRefitCore.response` | `GamlssRefitCore.response` — `response(self, q: Any, eta: Any) -> Any:` |
| `GamlssRefitCore.to_json` | `GamlssRefitCore.to_json` — `to_json(self) -> Any:` |
| `LearnerStore` | `LearnerStore` |
| `LearnerStore.basis_info_len` | `LearnerStore.basis_info_len` — `basis_info_len(self) -> Any:` |
| `LearnerStore.compute_aic` | `LearnerStore.compute_aic` — `compute_aic(self, selected: Any, risk: Any, sumw: Any, y_ss: Any, method: Any = "corrected", k: Any = 2.0, df: Any = None, coefs: Any = None) -> Any:` |
| `LearnerStore.compute_hat_diag_py` | `LearnerStore.compute_hat_diag_py` — `compute_hat_diag_py(self, selected: Any, coefs: Any = None) -> Any:` |
| `LearnerStore.compute_hat_weighted` | `LearnerStore.compute_hat_weighted` — `compute_hat_weighted(self, selected: Any, step_fits: Any, family: Any, coefs: Any = None) -> Any:` |
| `LearnerStore.concurvity` | `LearnerStore.concurvity` — `concurvity(self, selected: Any, coefs_vecs: Any) -> Any:` |
| `LearnerStore.extract_design` | `LearnerStore.extract_design` — `extract_design(self, which: Any) -> Any:` |
| `LearnerStore.extract_df` | `LearnerStore.extract_df` — `extract_df(self, which: Any) -> Any:` |
| `LearnerStore.extract_df_mboost` | `LearnerStore.extract_df_mboost` — `extract_df_mboost(self, which: Any) -> Any:` |
| `LearnerStore.extract_lambda` | `LearnerStore.extract_lambda` — `extract_lambda(self, which: Any) -> Any:` |
| `LearnerStore.extract_penalty` | `LearnerStore.extract_penalty` — `extract_penalty(self, which: Any) -> Any:` |
| `LearnerStore.hat_traces` | `LearnerStore.hat_traces` — `hat_traces(self) -> Any:` |
| `LearnerStore.learner_names` | `LearnerStore.learner_names` — `learner_names(self) -> Any:` |
| `LearnerStore.penalty_drift` | `LearnerStore.penalty_drift` — `penalty_drift(self, selected: Any) -> Any:` |
| `LearnerStore.predict` | `LearnerStore.predict` — `predict(self, offset: Any, selected: Any, coefs_vecs: Any, mstop: Any = None) -> Any:` |
| `LearnerStore.predict_newdata` | `LearnerStore.predict_newdata` — `predict_newdata(self, offset: Any, selected: Any, coefs_vecs: Any, x_data: Any, mstop: Any = None, by_data: Any = None, unseen_levels: Any = None) -> Any:` |
| `LearnerStore.predict_newdata_grid` | `LearnerStore.predict_newdata_grid` — `predict_newdata_grid(self, offset: Any, selected: Any, coefs_vecs: Any, x_data: Any, mstops: Any, by_data: Any = None, unseen_levels: Any = None) -> Any:` |
| `LearnerStore.predict_newdata_grouped` | `LearnerStore.predict_newdata_grouped` — `predict_newdata_grouped(self, offset: Any, selected: Any, coefs_vecs: Any, x_data: Any, mstop: Any = None, by_data: Any = None, unseen_levels: Any = None, n_rows: Any = None) -> Any:` |
| `LearnerStore.predict_newdata_kronecker` | `LearnerStore.predict_newdata_kronecker` — `predict_newdata_kronecker(self, offset: Any, selected: Any, coefs_vecs: Any, x_data: Any, n_resp: Any, mstop: Any = None) -> Any:` |
| `LearnerStore.predict_over_grid` | `LearnerStore.predict_over_grid` — `predict_over_grid(self, offset: Any, selected: Any, coefs_vecs: Any, x_data: Any, by_data: Any, grid_len: Any, grid_x: Any, grid_by: Any, mstop: Any = None, unseen_levels: Any = None, n_rows: Any = None) -> Any:` |
| `LearnerStore.scalar_centering` | `LearnerStore.scalar_centering` — `scalar_centering(self, which: Any) -> Any:` |
| `LearnerStore.score_newdata_path` | `LearnerStore.score_newdata_path` — `score_newdata_path(self, offset: Any, selected: Any, coefs_vecs: Any, x_data: Any, y: Any, weights: Any, family_name: Any, family_params: Any, metric: Any, parameter: Any = None, by_data: Any = None, unseen_levels: Any = None) -> Any:` |
| `LearnerStore.set_basis_info` | `LearnerStore.set_basis_info` — `set_basis_info(self, scalar_center: Any, scalar_centering: Any, bbs_center: Any, bbs_differences: Any, bbs_n_knots: Any, bbs_degree: Any, bbs_boundary_knots: Any, bbs_cyclic: Any, bbs_deriv: Any) -> Any:` |
| `LearnerStore.staged_held_out_risk` | `LearnerStore.staged_held_out_risk` — `staged_held_out_risk(self, offset: Any, selected: Any, coefs_vecs: Any, family_name: Any, family_params: Any, y: Any, w: Any) -> Any:` |
| `aggregate_split_p` | `aggregate_split_p` — `aggregate_split_p(p_values: Any, gamma: Any) -> Any:` |
| `auc_delong` | `auc_delong` — `auc_delong(observed: Any, scores: Any, weights: Any) -> Any:` |
| `augmented_scores` | `augmented_scores` — `augmented_scores(y: Any, a: Any, observed: Any, weights: Any, e_hat: Any, pi_hat: Any, mu0: Any, mu1: Any, estimand: Any, clip: Any = None, response_clip: Any = None) -> Any:` |
| `bbs_raw_design` | `bbs_raw_design` — `bbs_raw_design(x: Any, boundary_knots: Any, n_knots: Any, degree: Any) -> Any:` |
| `binomial_deviance` | `binomial_deviance` — `binomial_deviance(treatment: Any, probability: Any, weights: Any) -> Any:` |
| `boost_control` | `boost_control` — `boost_control(mstop: Any, nu: Any, risk: Any = "inbag", trace: Any = False) -> Any:` |
| `bootstrap_counts` | `bootstrap_counts` — `bootstrap_counts(flags: Any) -> Any:` |
| `bootstrap_percentile_interval` | `bootstrap_percentile_interval` — `bootstrap_percentile_interval(draws: Any, level: Any) -> Any:` |
| `brad_basis` | `brad_basis` — `brad_basis(x: Any, knots: Any = 100, theta: Any = None) -> Any:` |
| `brad_covariance` | `brad_covariance` — `brad_covariance(distance: Any, theta: Any) -> Any:` |
| `brad_design` | `brad_design` — `brad_design(x: Any, centres: Any, whitening: Any, theta: Any) -> Any:` |
| `brad_effective_range` | `brad_effective_range` — `brad_effective_range(x: Any, eps: Any = 1e-3) -> Any:` |
| `breslow_baseline` | `breslow_baseline` — `breslow_baseline(time: Any, event: Any, linpred_train: Any) -> Any:` |
| `breslow_curves` | `breslow_curves` — `breslow_curves(event_times: Any, n_event: Any, cumhaz: Any, center: Any, linpred_new: Any = None) -> Any:` |
| `breslow_horizon_survival` | `breslow_horizon_survival` — `breslow_horizon_survival(event_times: Any, n_event: Any, cumhaz: Any, center: Any, horizons: Any, linpred_new: Any) -> Any:` |
| `breslow_survfit` | `breslow_survfit` — `breslow_survfit(time: Any, event: Any, linpred_train: Any, linpred_new: Any = None) -> Any:` |
| `build_info` | `build_info` — `build_info() -> Any:` |
| `cell_nearest_distance` | `cell_nearest_distance` — `cell_nearest_distance(train_cells: Any, train_numeric: Any, new_cells: Any, new_numeric: Any) -> Any:` |
| `centered_group_effects` | `centered_group_effects` — `centered_group_effects(coef: Any, centering: Any) -> Any:` |
| `cluster_average` | `cluster_average` — `cluster_average(values: Any, weights: Any, clusters: Any, draws: Any = 1000, seed: Any = 0, level: Any = 0.95) -> Any:` |
| `coef_glmboost_py` | `coef_glmboost_py` — `coef_glmboost_py(selected: Any, coefs_flat: Any, nu: Any, n_learners: Any, centering_means: Any, off2int: Any, offset: Any, which: Any = None) -> Any:` |
| `column_diagnostics` | `column_diagnostics` — `column_diagnostics(x: Any, y: Any) -> Any:` |
| `concurvity_measures` | `concurvity_measures` — `concurvity_measures(x: Any, spans: Any, beta: Any) -> Any:` |
| `conformal_absolute_residuals` | `conformal_absolute_residuals` — `conformal_absolute_residuals(y: Any, predicted: Any) -> Any:` |
| `conformal_band_midpoint` | `conformal_band_midpoint` — `conformal_band_midpoint(lower: Any, upper: Any) -> Any:` |
| `conformal_calibration_count` | `conformal_calibration_count` — `conformal_calibration_count(n: Any, fraction: Any) -> Any:` |
| `conformal_clip_to_band` | `conformal_clip_to_band` — `conformal_clip_to_band(values: Any, lower: Any, upper: Any) -> Any:` |
| `conformal_count_cardinalities` | `conformal_count_cardinalities` — `conformal_count_cardinalities(lower: Any, upper: Any) -> Any:` |
| `conformal_count_default_score` | `conformal_count_default_score` — `conformal_count_default_score(family: Any, y: Any) -> Any:` |
| `conformal_count_rank_band` | `conformal_count_rank_band` — `conformal_count_rank_band(predicted: Any, zero_probability: Any, q: Any, uniforms: Any, level: Any) -> Any:` |
| `conformal_count_rank_scores` | `conformal_count_rank_scores` — `conformal_count_rank_scores(y: Any, predicted: Any, zero_probability: Any, uniforms: Any, level: Any) -> Any:` |
| `conformal_count_reference` | `conformal_count_reference` — `conformal_count_reference(y: Any, predicted: Any, weights: Any, zero_inflated: Any) -> Any:` |
| `conformal_count_uniforms` | `conformal_count_uniforms` — `conformal_count_uniforms(n: Any, seed: Any = None, stream: Any = 0, offset: Any = 0) -> Any:` |
| `conformal_cv_plus_bounds` | `conformal_cv_plus_bounds` — `conformal_cv_plus_bounds(residuals: Any, fold_of: Any, fold_predictions: Any, level: Any) -> Any:` |
| `conformal_cv_plus_guaranteed_coverage` | `conformal_cv_plus_guaranteed_coverage` — `conformal_cv_plus_guaranteed_coverage(n: Any, k: Any, level: Any) -> Any:` |
| `conformal_cv_plus_ranks` | `conformal_cv_plus_ranks` — `conformal_cv_plus_ranks(n: Any, level: Any) -> Any:` |
| `conformal_location_scale_band` | `conformal_location_scale_band` — `conformal_location_scale_band(mu: Any, sigma: Any, q: Any) -> Any:` |
| `conformal_location_scale_scores` | `conformal_location_scale_scores` — `conformal_location_scale_scores(y: Any, mu: Any, sigma: Any) -> Any:` |
| `conformal_min_calibration_size` | `conformal_min_calibration_size` — `conformal_min_calibration_size(level: Any) -> Any:` |
| `conformal_ordered_pair` | `conformal_ordered_pair` — `conformal_ordered_pair(a: Any, b: Any) -> Any:` |
| `conformal_poisson_band` | `conformal_poisson_band` — `conformal_poisson_band(predicted: Any, level: Any, q: Any, score: Any) -> Any:` |
| `conformal_poisson_scores` | `conformal_poisson_scores` — `conformal_poisson_scores(y: Any, predicted: Any, level: Any, score: Any) -> Any:` |
| `conformal_quantile` | `conformal_quantile` — `conformal_quantile(scores: Any, level: Any) -> Any:` |
| `conformal_quantile_scores` | `conformal_quantile_scores` — `conformal_quantile_scores(lower: Any, upper: Any, y: Any) -> Any:` |
| `conformal_rank_scale_columns` | `conformal_rank_scale_columns` — `conformal_rank_scale_columns(columns: Any, target: Any, keep: Any) -> Any:` |
| `conformal_scale_target` | `conformal_scale_target` — `conformal_scale_target(y: Any, predicted: Any) -> Any:` |
| `conformal_symmetric_band` | `conformal_symmetric_band` — `conformal_symmetric_band(predicted: Any, q: Any) -> Any:` |
| `conformal_tuned_mstop_choice` | `conformal_tuned_mstop_choice` — `conformal_tuned_mstop_choice(optimal: Any) -> Any:` |
| `conformal_tuned_mstop_next` | `conformal_tuned_mstop_next` — `conformal_tuned_mstop_next(largest: Any, cap: Any) -> Any:` |
| `conformal_tuned_mstop_start` | `conformal_tuned_mstop_start` — `conformal_tuned_mstop_start(model_mstop: Any) -> Any:` |
| `conformal_widen_band` | `conformal_widen_band` — `conformal_widen_band(lower: Any, upper: Any, q: Any) -> Any:` |
| `conformal_worst_covariate_bin` | `conformal_worst_covariate_bin` — `conformal_worst_covariate_bin(x: Any, lower: Any, upper: Any, y: Any, level: Any, bins: Any) -> Any:` |
| `contribution_shares` | `contribution_shares` — `contribution_shares(part: Any, rest: Any) -> Any:` |
| `crps_cdf_quadrature` | `crps_cdf_quadrature` — `crps_cdf_quadrature(cdf_below: Any, cdf_above: Any, step_below: Any, step_above: Any) -> Any:` |
| `crps_discrete_sum` | `crps_discrete_sum` — `crps_discrete_sum(y: Any, cdf: Any) -> Any:` |
| `crps_normal` | `crps_normal` — `crps_normal(y: Any, mu: Any, sigma: Any) -> Any:` |
| `cv_weights` | `cv_weights` — `cv_weights(n: Any, fold_type: Any, b: Any, prob: Any, seed: Any, weights: Any = None) -> Any:` |
| `cvrisk_foldwise` | `cvrisk_foldwise` — `cvrisk_foldwise(family_name: Any, family_params: Any, n_folds: Any, mstop: Any, provider: Any) -> Any:` |
| `cvrisk_gamboost` | `cvrisk_gamboost` — `cvrisk_gamboost(scalar_x: Any, scalar_names: Any, scalar_lambda: Any, scalar_center: Any, scalar_df: Any, bbs_x: Any, bbs_names: Any, bbs_knots: Any, bbs_degree: Any, bbs_differences: Any, bbs_df: Any, bbs_lambda: Any, bbs_center: Any, bbs_cyclic: Any, bbs_boundary_knots_lo: Any, bbs_boundary_knots_hi: Any, bbs_deriv: Any, bmono_x: Any, bmono_names: Any, bmono_knots: Any, bmono_degree: Any, bmono_differences: Any, bmono_df: Any, bmono_constraint: Any, bmono_type: Any, bmono_lambda2: Any, bmono_niter: Any, bmono_center: Any, btree_x: Any, btree_names: Any, btree_minbucket: Any, btree_minsplit: Any, btree_maxdepth: Any, y: Any, weights: Any, mstop: Any, nu: Any, family_name: Any, family_params: Any, folds: Any, judge: Any = None, offset: Any = None, scalar_by: Any = None, bbs_by: Any = None, bmono_by: Any = None, btree_by: Any = None, bmrf_levels: Any = None, bmrf_names: Any = None, bmrf_k: Any = None, bmrf_df: Any = None, bmrf_lambda: Any = None, bmrf_center: Any = None, bmrf_by: Any = None, bspatial_x1: Any = None, bspatial_x2: Any = None, bspatial_names: Any = None, bspatial_knots: Any = None, bspatial_degree: Any = None, bspatial_differences: Any = None, bspatial_df: Any = None, bspatial_lambda: Any = None, bspatial_center: Any = None, bspatial_by: Any = None, bkernel_x: Any = None, bkernel_names: Any = None, bkernel_bandwidth: Any = None, bkernel_df: Any = None, bkernel_kernel: Any = None, bkernel_center: Any = None, bkernel_by: Any = None, brandom_x: Any = None, brandom_names: Any = None, brandom_n_levels: Any = None, brandom_df: Any = None, brandom_lambda: Any = None, brandom_center: Any = None, brandom_by: Any = None, bmono_C: Any = None, buser_X: Any = None, buser_K: Any = None, buser_names: Any = None, buser_df: Any = None, buser_lambda: Any = None, buser_center: Any = None, buser_by: Any = None, risk: Any = "inbag", trace: Any = False, parallel: Any = True, fold_label: Any = "bootstrap", step: Any = "fixed", step_iterations: Any = 4) -> Any:` |
| `cvrisk_gamboost_specs` | `cvrisk_gamboost_specs` — `cvrisk_gamboost_specs(learner_specs: Any, y: Any, weights: Any, mstop: Any, nu: Any, family_name: Any, family_params: Any, folds: Any, judge: Any = None, offset: Any = None, risk: Any = "inbag", trace: Any = False, parallel: Any = True, fold_label: Any = "bootstrap", multinomial_n_classes: Any = None, step: Any = "fixed", step_iterations: Any = 4) -> Any:` |
| `cvrisk_gamboost_specs_extending` | `cvrisk_gamboost_specs_extending` — `cvrisk_gamboost_specs_extending(learner_specs: Any, y: Any, weights: Any, mstop: Any, nu: Any, family_name: Any, family_params: Any, folds: Any, judge: Any = None, offset: Any = None, risk: Any = "inbag", trace: Any = False, parallel: Any = True, fold_label: Any = "bootstrap", multinomial_n_classes: Any = None, step: Any = "fixed", step_iterations: Any = 4, max_mstop: Any = 10000, patience: Any = None, boundary_share: Any = 0.02, max_extensions: Any = None) -> Any:` |
| `cvrisk_gamboostlss` | `cvrisk_gamboostlss` — `cvrisk_gamboostlss(bbs_x: Any, param_nlearners: Any, bbs_knots: Any, bbs_degree: Any, bbs_differences: Any, bbs_df: Any, bbs_center: Any, learner_names: Any, y: Any, weights: Any, mstop: Any, nu: Any, family_name: Any, folds: Any, grid: Any, judge: Any = None, stabilization: Any = "none", nu_per_param: Any = None, step: Any = "fixed", step_iterations: Any = 4, response_scale: Any = None) -> Any:` |
| `cvrisk_gamboostlss_mixed` | `cvrisk_gamboostlss_mixed` — `cvrisk_gamboostlss_mixed(learner_specs: Any, y: Any, weights: Any, mstop: Any, nu: Any, family_name: Any, folds: Any, grid: Any, judge: Any = None, stabilization: Any = "none", nu_per_param: Any = None, step: Any = "fixed", step_iterations: Any = 4, response_scale: Any = None) -> Any:` |
| `cvrisk_glmboost` | `cvrisk_glmboost` — `cvrisk_glmboost(x_matrix: Any, y: Any, weights: Any, mstop: Any, nu: Any, family_name: Any, family_params: Any, learner_names: Any, folds: Any, judge: Any = None, offset: Any = None, risk: Any = "inbag", trace: Any = False, parallel: Any = True, fold_label: Any = "bootstrap", step: Any = "fixed", step_iterations: Any = 4, intercept: Any = True, center: Any = True) -> Any:` |
| `cvrisk_glmboostlss` | `cvrisk_glmboostlss` — `cvrisk_glmboostlss(x_all: Any, param_ncols: Any, learner_names: Any, y: Any, weights: Any, mstop: Any, nu: Any, family_name: Any, folds: Any, grid: Any, stabilization: Any = "none", nu_per_param: Any = None, step: Any = "fixed", step_iterations: Any = 4, response_scale: Any = None) -> Any:` |
| `cvrisk_selected_gamboost_specs` | `cvrisk_selected_gamboost_specs` — `cvrisk_selected_gamboost_specs(learner_specs: Any, y: Any, weights: Any, mstop: Any, nu: Any, family_name: Any, family_params: Any, folds: Any, offset: Any = None, multinomial_n_classes: Any = None, step: Any = "fixed", step_iterations: Any = 4) -> Any:` |
| `cvrisk_selected_glmboost` | `cvrisk_selected_glmboost` — `cvrisk_selected_glmboost(x_matrix: Any, y: Any, weights: Any, mstop: Any, nu: Any, family_name: Any, family_params: Any, learner_names: Any, folds: Any, offset: Any = None, step: Any = "fixed", step_iterations: Any = 4, intercept: Any = True, center: Any = True) -> Any:` |
| `declaration_quantiles` | `declaration_quantiles` — `declaration_quantiles(values: Any, probs: Any) -> Any:` |
| `declaration_shape_check` | `declaration_shape_check` — `declaration_shape_check(sign: Any, second_difference: Any, grid: Any, risk_constrained: Any, risk_unconstrained: Any) -> Any:` |
| `declaration_sign_check` | `declaration_sign_check` — `declaration_sign_check(sign: Any, hi: Any, lo: Any, fold_hi: Any, fold_lo: Any, boot: Any) -> Any:` |
| `deselect_group_sums` | `deselect_group_sums` — `deselect_group_sums(values: Any, lengths: Any) -> Any:` |
| `deselect_keep` | `deselect_keep` — `deselect_keep(shares: Any, threshold: Any) -> Any:` |
| `deselect_lss_shares` | `deselect_lss_shares` — `deselect_lss_shares(parameter: Any, learner: Any, risk_change: Any) -> Any:` |
| `disc_scan` | `disc_scan` — `disc_scan(u: Any, background: Any, target: Any, fit: Any, cuts: Any, min_side: Any) -> Any:` |
| `disc_step_z` | `disc_step_z` — `disc_step_z(background: Any, indicator: Any, target: Any, eval: Any) -> Any:` |
| `discovery_learning_rate` | `discovery_learning_rate` — `discovery_learning_rate(family: Any) -> Any:` |
| `discovery_mean_step` | `discovery_mean_step` — `discovery_mean_step(family: Any) -> Any:` |
| `draw_contrast` | `draw_contrast` — `draw_contrast(alternative: Any, reference: Any, draws_alternative: Any, draws_reference: Any, level: Any) -> Any:` |
| `draw_p` | `draw_p` — `draw_p(draws: Any) -> Any:` |
| `effect_gates` | `effect_gates` — `effect_gates(score: Any, weights: Any, groups: Any, n_groups: Any) -> Any:` |
| `effect_holm_adjust` | `effect_holm_adjust` — `effect_holm_adjust(p: Any) -> Any:` |
| `effect_linear_predict` | `effect_linear_predict` — `effect_linear_predict(x: Any, p: Any, beta: Any) -> Any:` |
| `effect_robust_wls` | `effect_robust_wls` — `effect_robust_wls(y: Any, x: Any, p: Any, weights: Any) -> Any:` |
| `effect_wald_test` | `effect_wald_test` — `effect_wald_test(coefficients: Any, covariance: Any, p: Any, indices: Any) -> Any:` |
| `effective_sample_size` | `effective_sample_size` — `effective_sample_size(weights: Any) -> Any:` |
| `family_class` | `family_class` — `family_class(family_name: Any, family_params: Any, f: Any) -> Any:` |
| `family_diagnostics` | `family_diagnostics` — `family_diagnostics(name: Any, params: Any, y: Any, f: Any, w: Any, nuisance: Any = None, nuisance_vec: Any = None) -> Any:` |
| `family_held_out_rows` | `family_held_out_rows` — `family_held_out_rows(family: Any, params: Any, y: Any, eta: Any, nuisance: Any = None, nuisance_vec: Any = None) -> Any:` |
| `family_held_out_score` | `family_held_out_score` — `family_held_out_score(family: Any, params: Any, y: Any, eta: Any, weights: Any, nuisance: Any = None, nuisance_vec: Any = None) -> Any:` |
| `fit_gamboost` | `fit_gamboost` — `fit_gamboost(scalar_x: Any, scalar_names: Any, scalar_lambda: Any, scalar_center: Any, scalar_df: Any, bbs_x: Any, bbs_names: Any, bbs_knots: Any, bbs_degree: Any, bbs_differences: Any, bbs_df: Any, bbs_lambda: Any, bbs_center: Any, bbs_cyclic: Any, bbs_boundary_knots_lo: Any, bbs_boundary_knots_hi: Any, bbs_deriv: Any, bmono_x: Any, bmono_names: Any, bmono_knots: Any, bmono_degree: Any, bmono_differences: Any, bmono_df: Any, bmono_constraint: Any, bmono_type: Any, bmono_lambda2: Any, bmono_niter: Any, bmono_center: Any, btree_x: Any, btree_names: Any, btree_minbucket: Any, btree_minsplit: Any, btree_maxdepth: Any, bmrf_levels: Any, bmrf_names: Any, bmrf_k: Any, bmrf_df: Any, bmrf_lambda: Any, bmrf_center: Any, bspatial_x1: Any, bspatial_x2: Any, bspatial_names: Any, bspatial_knots: Any, bspatial_degree: Any, bspatial_differences: Any, bspatial_df: Any, bspatial_lambda: Any, bspatial_center: Any, y: Any, weights: Any, mstop: Any, nu: Any, family_name: Any, family_params: Any, offset: Any = None, scalar_by: Any = None, bbs_by: Any = None, bmono_by: Any = None, btree_by: Any = None, bmrf_by: Any = None, bspatial_by: Any = None, bkernel_x: Any = None, bkernel_names: Any = None, bkernel_bandwidth: Any = None, bkernel_df: Any = None, bkernel_kernel: Any = None, bkernel_center: Any = None, bkernel_by: Any = None, brandom_x: Any = None, brandom_names: Any = None, brandom_n_levels: Any = None, brandom_df: Any = None, brandom_lambda: Any = None, brandom_center: Any = None, brandom_by: Any = None, bmono_C: Any = None, buser_X: Any = None, buser_K: Any = None, buser_names: Any = None, buser_df: Any = None, buser_lambda: Any = None, buser_center: Any = None, buser_by: Any = None, multinomial_n_classes: Any = None, risk: Any = "inbag", trace: Any = False, component_fraction: Any = None, subsample_seed: Any = 0, step: Any = "fixed", step_iterations: Any = 4) -> Any:` |
| `fit_gamboost_continue_specs` | `fit_gamboost_continue_specs` — `fit_gamboost_continue_specs(learner_specs: Any, y: Any, weights: Any, mstop: Any, nu: Any, family_name: Any, family_params: Any, warm_fitted: Any, warm_offset: Any, warm_risk: Any, warm_selected: Any, warm_coefs: Any, warm_residual: Any, step: Any = "fixed", step_iterations: Any = 4, warm_nuisance: Any = None) -> Any:` |
| `fit_gamboost_specs` | `fit_gamboost_specs` — `fit_gamboost_specs(learner_specs: Any, y: Any, weights: Any, mstop: Any, nu: Any, family_name: Any, family_params: Any, offset: Any = None, multinomial_n_classes: Any = None, risk: Any = "inbag", trace: Any = False, component_fraction: Any = None, subsample_seed: Any = 0, step: Any = "fixed", step_iterations: Any = 4) -> Any:` |
| `fit_gamboostlss` | `fit_gamboostlss` — `fit_gamboostlss(bbs_x: Any, param_nlearners: Any, bbs_knots: Any, bbs_degree: Any, bbs_differences: Any, bbs_df: Any, bbs_center: Any, learner_names: Any, y: Any, weights: Any, mstop: Any, nu: Any, family_name: Any, offsets: Any = None, mstop_per_param: Any = None, trace: Any = False, stabilization: Any = "none", nu_per_param: Any = None, step: Any = "fixed", step_iterations: Any = 4, response_scale: Any = None) -> Any:` |
| `fit_gamboostlss_mixed` | `fit_gamboostlss_mixed` — `fit_gamboostlss_mixed(learner_specs: Any, y: Any, weights: Any, mstop: Any, nu: Any, family_name: Any, offsets: Any = None, mstop_per_param: Any = None, method: Any = "cyclic", trace: Any = False, stabilization: Any = "none", nu_per_param: Any = None, step: Any = "fixed", step_iterations: Any = 4, response_scale: Any = None) -> Any:` |
| `fit_glmboost` | `fit_glmboost` — `fit_glmboost(x_matrix: Any, y: Any, weights: Any, mstop: Any, nu: Any, family_name: Any, family_params: Any, learner_names: Any, offset: Any = None, risk: Any = "inbag", trace: Any = False, component_fraction: Any = None, subsample_seed: Any = 0, center: Any = True, step: Any = "fixed", step_iterations: Any = 4, intercept: Any = True) -> Any:` |
| `fit_glmboostlss` | `fit_glmboostlss` — `fit_glmboostlss(x_all: Any, param_ncols: Any, learner_names: Any, y: Any, weights: Any, mstop: Any, nu: Any, family_name: Any, offsets: Any = None, mstop_per_param: Any = None, trace: Any = False, stabilization: Any = "none", nu_per_param: Any = None, step: Any = "fixed", step_iterations: Any = 4, response_scale: Any = None) -> Any:` |
| `fit_glmboostlss_noncyclic` | `fit_glmboostlss_noncyclic` — `fit_glmboostlss_noncyclic(x_all: Any, param_ncols: Any, learner_names: Any, y: Any, weights: Any, mstop: Any, nu: Any, family_name: Any, offsets: Any = None, trace: Any = False, stabilization: Any = "none", nu_per_param: Any = None, step: Any = "fixed", step_iterations: Any = 4, response_scale: Any = None) -> Any:` |
| `gam_refit_bootstrap_core` | `gam_refit_bootstrap_core` — `gam_refit_bootstrap_core(names: Any, designs: Any, penalties: Any, bys: Any, y: Any, counts: Any, weights: Any = None, variables: Any = None) -> Any:` |
| `gam_refit_core` | `gam_refit_core` — `gam_refit_core(names: Any, designs: Any, penalties: Any, bys: Any, y: Any, weights: Any = None, offset: Any = None, family: Any = "Gaussian", theta: Any = None, time: Any = None, variables: Any = None, gram: Any = False) -> Any:` |
| `gam_refit_core_from_json` | `gam_refit_core_from_json` — `gam_refit_core_from_json(state: Any) -> Any:` |
| `gamlss_refit_core` | `gamlss_refit_core` — `gamlss_refit_core(family: Any, param_names: Any, names: Any, designs: Any, penalties: Any, bys: Any, y: Any, weights: Any = None, start_etas: Any = None, variables: Any = None) -> Any:` |
| `gamlss_refit_core_from_json` | `gamlss_refit_core_from_json` — `gamlss_refit_core_from_json(family: Any, state: Any) -> Any:` |
| `gcm_test` | `gcm_test` — `gcm_test(r_x: Any, r_y: Any) -> Any:` |
| `gcm_test_multi` | `gcm_test_multi` — `gcm_test_multi(r_x: Any, k: Any, r_y: Any) -> Any:` |
| `glmboost_design` | `glmboost_design` — `glmboost_design(x: Any, centering_means: Any, intercept: Any) -> Any:` |
| `group_bootstrap_counts` | `group_bootstrap_counts` — `group_bootstrap_counts(codes: Any, eligible: Any, b: Any, seed: Any) -> Any:` |
| `group_components` | `group_components` — `group_components(codes: Any) -> Any:` |
| `h_statistic` | `h_statistic` — `h_statistic(pd_jk: Any, pd_j: Any, pd_k: Any) -> Any:` |
| `impute_model_fit` | `impute_model_fit` — `impute_model_fit(target: Any, x: Any, levels: Any) -> Any:` |
| `impute_model_predict` | `impute_model_predict` — `impute_model_predict(params: Any, x: Any) -> Any:` |
| `influence_variance` | `influence_variance` — `influence_variance(influence: Any, weights: Any, groups: Any = None) -> Any:` |
| `inverse_response_transform` | `inverse_response_transform` — `inverse_response_transform(values: Any, kind: Any) -> Any:` |
| `ipcw_auc_at` | `ipcw_auc_at` — `ipcw_auc_at(time: Any, event: Any, weights: Any, score: Any, horizon: Any, censoring_floor: Any) -> Any:` |
| `ipcw_brier_at` | `ipcw_brier_at` — `ipcw_brier_at(time: Any, event: Any, weights: Any, survival: Any, horizon: Any, censoring_floor: Any) -> Any:` |
| `iv_ratio` | `iv_ratio` — `iv_ratio(y_res: Any, d_res: Any, z_res: Any) -> Any:` |
| `iv_ratio_weighted` | `iv_ratio_weighted` — `iv_ratio_weighted(y_res: Any, d_res: Any, z_res: Any, weights: Any) -> Any:` |
| `joint_support` | `joint_support` — `joint_support(train: Any, new: Any) -> Any:` |
| `kron` | `kron` — `kron(a: Any, b: Any) -> Any:` |
| `leftover_share` | `leftover_share` — `leftover_share(y: Any, fitted: Any) -> Any:` |
| `linear_dependence` | `linear_dependence` — `linear_dependence(columns: Any, n: Any, p: Any, tol: Any) -> Any:` |
| `log_residual_variance` | `log_residual_variance` — `log_residual_variance(y: Any, fitted: Any, weights: Any) -> Any:` |
| `log_variance_screen` | `log_variance_screen` — `log_variance_screen(residual: Any, design: Any) -> Any:` |
| `lss_auto_response_scale` | `lss_auto_response_scale` — `lss_auto_response_scale(family_name: Any, n_params: Any, y: Any, weights: Any) -> Any:` |
| `lss_family_diagnostics` | `lss_family_diagnostics` — `lss_family_diagnostics(name: Any, n_params: Any, y: Any, etas: Any, w: Any) -> Any:` |
| `lss_quantile` | `lss_quantile` — `lss_quantile(family: Any, columns: Any) -> Any:` |
| `lss_row_nll` | `lss_row_nll` — `lss_row_nll(name: Any, n_params: Any, y: Any, etas: Any) -> Any:` |
| `mean_and_iid_se` | `mean_and_iid_se` — `mean_and_iid_se(values: Any) -> Any:` |
| `median_repeat_combine` | `median_repeat_combine` — `median_repeat_combine(estimates: Any, ses: Any) -> Any:` |
| `modifier_scale` | `modifier_scale` — `modifier_scale(values: Any, tail: Any, skew_ratio: Any) -> Any:` |
| `normal_interval_p` | `normal_interval_p` — `normal_interval_p(estimate: Any, lower: Any, upper: Any, level: Any) -> Any:` |
| `paired_gain` | `paired_gain` — `paired_gain(before: Any, after: Any, partitions: Any = 1, threshold_se: Any = 2.0) -> Any:` |
| `paired_mean_se` | `paired_mean_se` — `paired_mean_se(first: Any, second: Any) -> Any:` |
| `percentile_interval` | `percentile_interval` — `percentile_interval(values: Any, level: Any, centre_rows: Any) -> Any:` |
| `predict_gamboost` | `predict_gamboost` — `predict_gamboost(offset: Any, selected: Any, coefs_vecs: Any, nu: Any, scalar_x: Any, bbs_x: Any, bbs_knots: Any, bbs_degree: Any, bbs_differences: Any, _bbs_df: Any, bbs_center: Any, scalar_center: Any, _scalar_lambda: Any, bbs_boundary_knots: Any, bbs_deriv: Any, scalar_centering_means: Any, bmono_x: Any, bmono_knots: Any, bmono_degree: Any, bmono_differences: Any, bmono_boundary_knots: Any, btree_x: Any, mstop: Any = None, scalar_by: Any = None, bbs_by: Any = None, bmono_by: Any = None, btree_by: Any = None, bmrf_levels: Any = None, bmrf_k: Any = None, bmrf_center: Any = None, bmrf_by: Any = None, bspatial_x1: Any = None, bspatial_x2: Any = None, bspatial_knots: Any = None, bspatial_degree: Any = None, bspatial_differences: Any = None, bspatial_center: Any = None, bspatial_by: Any = None, bspatial_boundary_knots: Any = None, brandom_x: Any = None, brandom_n_levels: Any = None, brandom_center: Any = None, brandom_by: Any = None, brandom_centering_means: Any = None) -> Any:` |
| `predict_glmboost` | `predict_glmboost` — `predict_glmboost(offset: Any, selected: Any, coefs_flat: Any, nu: Any, x_matrix: Any, mstop: Any = None, centering_means: Any = None) -> Any:` |
| `predict_glmboost_path` | `predict_glmboost_path` — `predict_glmboost_path(offset: Any, selected: Any, coefs_flat: Any, nu: Any, x_matrix: Any, grid: Any, centering_means: Any = None) -> Any:` |
| `preexplore_json` | `preexplore_json` — `preexplore_json(x: Any, y: Any, categorical: Any, transforms: Any, engine: Any = "native", seed: Any = 0, rounds: Any = 300, pairs: Any = True) -> Any:` |
| `prefer_simpler_cv` | `prefer_simpler_cv` — `prefer_simpler_cv(simple: Any, rich: Any, multiple: Any) -> Any:` |
| `prepare_gamboostlss_stores` | `prepare_gamboostlss_stores` — `prepare_gamboostlss_stores(learner_specs: Any, weights: Any, nu: Any, nu_per_param: Any = None) -> Any:` |
| `prepare_learner_store` | `prepare_learner_store` — `prepare_learner_store(learner_specs: Any, weights: Any, nu: Any) -> Any:` |
| `probe_permutations` | `probe_permutations` — `probe_permutations(n: Any, count: Any, seed: Any) -> Any:` |
| `probe_stop` | `probe_stop` — `probe_stop(selected: Any, is_shadow: Any) -> Any:` |
| `propodds_response` | `propodds_response` — `propodds_response(f: Any, sigma: Any) -> Any:` |
| `rearrange_levels` | `rearrange_levels` — `rearrange_levels(predictions: Any) -> Any:` |
| `residual_cluster_false_alarm_rate` | `residual_cluster_false_alarm_rate` — `residual_cluster_false_alarm_rate(clusters: Any, draws: Any = 200, permutations: Any = 199, seed: Any = 0, alpha: Any = 0.05) -> Any:` |
| `residual_cluster_icc` | `residual_cluster_icc` — `residual_cluster_icc(residuals: Any, clusters: Any, permutations: Any = 999, seed: Any = 0) -> Any:` |
| `residual_influence` | `residual_influence` — `residual_influence(residuals: Any, hat: Any, alpha: Any = 0.05) -> Any:` |
| `residual_influence_false_alarm_rate` | `residual_influence_false_alarm_rate` — `residual_influence_false_alarm_rate(hat: Any, sigma: Any, draws: Any = 200, seed: Any = 0, alpha: Any = 0.05) -> Any:` |
| `residual_serial_dependence` | `residual_serial_dependence` — `residual_serial_dependence(residuals: Any, times: Any, lags: Any, permutations: Any = 999, seed: Any = 0) -> Any:` |
| `residual_serial_false_alarm_rate` | `residual_serial_false_alarm_rate` — `residual_serial_false_alarm_rate(times: Any, lags: Any, draws: Any = 200, permutations: Any = 199, seed: Any = 0, alpha: Any = 0.05) -> Any:` |
| `response_score` | `response_score` — `response_score(y: Any, predicted: Any, weights: Any, metric: Any, parameter: Any = None) -> Any:` |
| `response_score_path` | `response_score_path` — `response_score_path(y: Any, predictions: Any, weights: Any, metric: Any, parameter: Any = None) -> Any:` |
| `risk_path_rises` | `risk_path_rises` — `risk_path_rises(family: Any, params: Any, risk: Any) -> Any:` |
| `rloss` | `rloss` — `rloss(y: Any, t: Any, m_hat: Any, e_hat: Any, tau_hat: Any, clip: Any) -> Any:` |
| `rloss_transform` | `rloss_transform` — `rloss_transform(y: Any, t: Any, m_hat: Any, e_hat: Any, clip: Any) -> Any:` |
| `rloss_transform_weighted` | `rloss_transform_weighted` — `rloss_transform_weighted(y: Any, t: Any, m_hat: Any, e_hat: Any, clip: Any, sample_weight: Any, observed: Any, response: Any, response_clip: Any = None) -> Any:` |
| `row_tensor` | `row_tensor` — `row_tensor(a: Any, b: Any) -> Any:` |
| `same_family_glm_fit` | `same_family_glm_fit` — `same_family_glm_fit(family: Any, y: Any, weights: Any, designs: Any, offsets: Any, starts: Any, nuisance: Any) -> Any:` |
| `screen_main_effects` | `screen_main_effects` — `screen_main_effects(x: Any, residual: Any, bins: Any, seed: Any, factor_levels: Any, top_k: Any = None) -> Any:` |
| `screen_pairs` | `screen_pairs` — `screen_pairs(x: Any, residual: Any, bins: Any, seed: Any, factor_levels: Any, pairs: Any, top_k: Any = None) -> Any:` |
| `search_fold_scores` | `search_fold_scores` — `search_fold_scores(advance: Any, ends: Any, grids: Any, boundary_share: Any) -> Any:` |
| `search_gamboostlss_grid` | `search_gamboostlss_grid` — `search_gamboostlss_grid(learner_specs: Any, y: Any, weights: Any, nu: Any, family_name: Any, folds: Any, ends: Any, grids: Any, boundary_share: Any, judge: Any = None, stabilization: Any = "none", nu_per_param: Any = None, step: Any = "fixed", step_iterations: Any = 4, response_scale: Any = None, method: Any = "cyclic") -> Any:` |
| `select_score_path` | `select_score_path` — `select_score_path(scores: Any) -> Any:` |
| `special_function` | `special_function` — `special_function(name: Any, args: Any) -> Any:` |
| `specification_curve_sign_test` | `specification_curve_sign_test` — `specification_curve_sign_test(scores: Any, selected: Any, partitions: Any = 1, max_draws: Any = 10000, seed: Any = 0) -> Any:` |
| `stabsel_d_rc` | `stabsel_d_rc` — `stabsel_d_rc(theta: Any, which: Any, b: Any, r: Any) -> Any:` |
| `stabsel_max_q` | `stabsel_max_q` — `stabsel_max_q(p: Any, b: Any) -> Any:` |
| `stabsel_min_d` | `stabsel_min_d` — `stabsel_min_d(q: Any, p: Any, cutoff: Any, b: Any) -> Any:` |
| `stabsel_params` | `stabsel_params` — `stabsel_params(p: Any, cutoff: Any = None, q: Any = None, pfer: Any = None, b: Any = 50, assumption: Any = "unimodal", sampling_type: Any = "SS") -> Any:` |
| `stabsel_selected_gamboostlss` | `stabsel_selected_gamboostlss` — `stabsel_selected_gamboostlss(bbs_x: Any, param_nlearners: Any, bbs_knots: Any, bbs_degree: Any, bbs_differences: Any, bbs_df: Any, bbs_center: Any, learner_names: Any, y: Any, weights: Any, mstop: Any, nu: Any, family_name: Any, folds: Any, stabilization: Any = "none", nu_per_param: Any = None, step: Any = "fixed", step_iterations: Any = 4, response_scale: Any = None) -> Any:` |
| `stabsel_selected_gamboostlss_mixed` | `stabsel_selected_gamboostlss_mixed` — `stabsel_selected_gamboostlss_mixed(learner_specs: Any, y: Any, weights: Any, mstop: Any, nu: Any, family_name: Any, folds: Any, q: Any = None, stabilization: Any = "none", nu_per_param: Any = None, step: Any = "fixed", step_iterations: Any = 4, response_scale: Any = None) -> Any:` |
| `stabsel_selected_glmboostlss` | `stabsel_selected_glmboostlss` — `stabsel_selected_glmboostlss(x_all: Any, param_ncols: Any, y: Any, weights: Any, mstop: Any, nu: Any, family_name: Any, folds: Any, q: Any = None, stabilization: Any = "none", nu_per_param: Any = None, step: Any = "fixed", step_iterations: Any = 4, response_scale: Any = None) -> Any:` |
| `stabsel_um_const` | `stabsel_um_const` — `stabsel_um_const(cutoff: Any, b: Any, theta: Any) -> Any:` |
| `student_t_pdf` | `student_t_pdf` — `student_t_pdf(y: Any, mu: Any, sigma: Any, df: Any) -> Any:` |
| `subgroup_scores` | `subgroup_scores` — `subgroup_scores(scores: Any, weights: Any, codes: Any, n_levels: Any, groups: Any = None) -> Any:` |
| `term_basis_design` | `term_basis_design` — `term_basis_design(recipe: Any, vars: Any, unseen_zero: Any = False) -> Any:` |
| `transform_ecdf_params` | `transform_ecdf_params` — `transform_ecdf_params(values: Any) -> Any:` |
| `transform_forward` | `transform_forward` — `transform_forward(kind: Any, params: Any, x: Any) -> Any:` |
| `transform_inverse` | `transform_inverse` — `transform_inverse(kind: Any, params: Any, v: Any) -> Any:` |
| `transform_jacobian` | `transform_jacobian` — `transform_jacobian(kind: Any, params: Any, y: Any) -> Any:` |
| `transform_mean_sd` | `transform_mean_sd` — `transform_mean_sd(values: Any) -> Any:` |
| `transform_median` | `transform_median` — `transform_median(values: Any) -> Any:` |
| `two_part_mean` | `two_part_mean` — `two_part_mean(employed: Any, log_mean: Any, log_variance: Any) -> Any:` |
| `varimp_attribution` | `varimp_attribution` — `varimp_attribution(changes: Any, groups: Any, n_groups: Any, percent: Any) -> Any:` |
| `winsorize_pseudo_outcome` | `winsorize_pseudo_outcome` — `winsorize_pseudo_outcome(pseudo: Any, weights: Any, tail: Any) -> Any:` |

